In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
import os
from scipy.stats import zscorets.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import validation_curve, learning_curve, StratifiedKFold, cross_val_score
from sklearn.metrics import (f1_score, accuracy_score, precision_score, recall_score,
                           roc_auc_score, average_precision_score, confusion_matrix,
                           roc_curve, precision_recall_curve, classification_report)
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                            ExtraTreesClassifier, AdaBoostClassifier, BaggingClassifier,
                            VotingClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score, validation_curve
from sklearn.metrics import classification_report
import pickle


### Importing the file

In [ ]:
df = pd.read_csv('chronickidneydisease.csv')
df.head()

### Initial Information About the Dataframe

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

### Creating Copy of df For Further Operations

In [ ]:
df_clean=df.copy()


### Looking at the State of Non-Numerical Columns

In [ ]:
non_numerical_columns=df.select_dtypes(exclude=['number']).columns.tolist()
for column in non_numerical_columns:
    print(f"Column: {column}\n")
    print(f"Unique Values:\n {df[column].unique()}\n")
    print(f"Value Counts:\n {df[column].value_counts()}\n")
    print(f"Missing Values:\n {df[column].isnull().sum()}\n")
    print(f"Data Type:\n {df[column].dtype}\n")
    print(f"Description:\n {df[column].describe()}\n")
    print("--------------------------------\n\n")

- Found some numerical columns stored as objects. Will rectify next.

### Cleaning all Numerical Columns (\t ' ' ? and Other Issues) And Changing Incorrect Datatypes

In [ ]:
# Function to clean numerical columns
def clean_numerical_column(series):
    # Remove tab characters and convert to string
    series = series.astype(str).str.replace('\t', '')

    # Replace '?' with NaN
    series = series.replace('?', np.nan)

    # Convert to numeric, coercing errors to NaN, if in string format
    series = pd.to_numeric(series, errors='coerce')

    return series

# Clean the numerical columns stored as objects
numerical_columns_to_clean = ['pcv', 'wc', 'rc']
for col in numerical_columns_to_clean:
    df_clean[col] = clean_numerical_column(df_clean[col])

    # Print summary after cleaning
    print(f"\nColumn: {col}")
    print(f"Data type after cleaning: {df_clean[col].dtype}")
    print(f"Number of missing values: {df_clean[col].isnull().sum()}")
    print(f"Value range: {df_clean[col].min()} to {df_clean[col].max()}")
    print(f"Mean: {df_clean[col].mean():.2f}")
    print(f"Median: {df_clean[col].median():.2f}")
    print("--------------------------------")

### Cleaning Categorical Columns (Similar Issues)

In [ ]:
unclean_categorical_columns=['classification','cad','dm']

def clean_categorical_column(series):
    series = series.astype(str).str.replace('\t','')
    series = series.astype(str).str.replace(' ', '')
    series = series.replace('?', np.nan)
    series= series.replace('nan',np.nan)
    return series

for column in unclean_categorical_columns:
    df_clean[column]=clean_categorical_column(df_clean[column])

    print(f"Column: {column}")
    print(f"Unique Values:\n {df_clean[column].unique()}\n")
    print(f"Value Counts:\n {df_clean[column].value_counts()}\n")
    print(f"Missing Values:\n {df_clean[column].isnull().sum()}\n")
    print(f"Data Type:\n {df_clean[column].dtype}\n")
    print(f"Description:\n {df_clean[column].describe()}\n")
    print("--------------------------------\n\n")

### Checking All Numerical Columns and Their Current State

In [ ]:
numerical_columns = df.select_dtypes(include=['number']).columns.tolist()

for column in numerical_columns:
    print(f"Column: {column}")
    print(f"Missing Values:\n {df[column].isnull().sum()}\n")
    print(f"Data Type:\n {df[column].dtype}\n")
    print(f"Description:\n {df[column].describe()}\n")
    print("--------------------------------\n\n")

### Missing Pattern Analysis For Integrity of Data

In [ ]:
# Set the style for better visualization
sns.set_palette("husl")

In [ ]:

# 1. Create a missing value heatmap
def plot_missing_heatmap(df):
    # Calculate missing values percentage for each column
    missing_percentage = (df.isnull().sum() / len(df)) * 100

    # Create a figure with two subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))

    # Plot 1: Missing value heatmap
    sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis', ax=ax1)
    ax1.set_title('Missing Values Heatmap', pad=20)

    # Plot 2: Missing values percentage bar plot
    missing_percentage.sort_values(ascending=False).plot(kind='bar', ax=ax2)
    ax2.set_title('Percentage of Missing Values by Column', pad=20)
    ax2.set_ylabel('Percentage Missing')
    plt.xticks(rotation=45, ha='right')

    plt.tight_layout()
    plt.show()

    # Print detailed missing value statistics
    print("\nDetailed Missing Value Statistics:")
    print("-" * 50)
    for col in df.columns:
        missing = df[col].isnull().sum()
        percentage = (missing / len(df)) * 100
        print(f"{col}:")
        print(f"  Missing values: {missing} ({percentage:.2f}%)")
        if df[col].dtype == 'object':
            print(f"  Most common value: {df[col].value_counts().index[0] if not df[col].value_counts().empty else 'N/A'}")
        else:
            print(f"  Mean: {df[col].mean():.2f}")
            print(f"  Median: {df[col].median():.2f}")
        print("-" * 50)

# 2. Analyze patterns in missing values
def analyze_missing_patterns(df):
    # Create a correlation matrix of missing values
    missing_corr = df.isnull().corr()

    # Plot the correlation heatmap with adjusted settings
    plt.figure(figsize=(15, 12))  # Increased figure size
    sns.heatmap(missing_corr,
                annot=True,
                cmap='coolwarm',
                center=0,
                annot_kws={'size': 8},  # Smaller annotation font
                fmt='.2f')  # Format to 2 decimal places

    plt.title('Correlation of Missing Values Between Columns', pad=20)
    plt.xticks(rotation=45, ha='right', fontsize=8)  # Rotated and smaller x-tick labels
    plt.yticks(fontsize=8)  # Smaller y-tick labels
    plt.tight_layout()
    plt.show()

    # Analyze if missing values are related to the target variable
    if 'classification' in df.columns:
        print("\nMissing Values by Target Variable:")
        print("-" * 70)  # Wider separator
        for col in df.columns:
            if df[col].isnull().any():
                missing_by_target = df.groupby('classification')[col].apply(lambda x: x.isnull().mean())
                print(f"\n{col:15}")  # Fixed width for column names
                print("-" * 30)
                for target, value in missing_by_target.items():
                    print(f"{target:10}: {value:.2%}")  # Format as percentage
                print("-" * 70)


In [ ]:
# Run the analysis
print("Step 1: Visualizing Missing Values")
plot_missing_heatmap(df_clean)

print("\nStep 2: Analyzing Missing Value Patterns")
analyze_missing_patterns(df_clean)



## Null value heatmap: [Visually Dense/Populated]
1. rbc
2. sod, pot (related nulls)
3. wc,rc
4. wc
5. pcv
6. sg,al,su,rbc,pc
7. bgr
8. bp
9. age

## Percentage missing values
- \>30% [CRITICAL]
[rbc,rc]
might need special handling or consideration for dropping

- 5-30% [MODERATE]
[sg,al,su,pc,bgr,sod,pt,hemo,pcv,wc]

- <5% [MINOR]
[id, age, bp, pcc, ba, bu, sc, htn, dm, cad, appet, pe, ane, classification]
simple imputation

## Missing Values Correlation Heatmap Analysis
- Strong positive correlations (values close to 1): Indicates columns that tend to be missing together (darker shades of red)
- Strong negative correlations (values close to -1): Indicates columns that tend to be present when others are missing (darker shades of blue (ABSENT))
- Values close to 0: Indicates independent missing patterns

- SP: [{appet,pe,ane}, {htn,dm,cad}, {pcc,ba}, {sod,pot}, {sg,al,su}, {bu,sc}, {wc,rc}, {hemo,pcv}, {hemo,pcv,wc,rc}, {sg,al,su,rbc,pc}]
- ModerateP: [{bgr,bu,sc,sod,pot}, {sod,pot,hemo,pcv,wc,rc}]
- WP: [{sg,al,su,rbc,pc}, {hemo,pcv,wc,rc}]

---

### Detailed Analysis With Medical Significance:

#### **Strong Positive Groups (SP)**
- **{appet, pe, ane}:** All are symptoms or clinical findings. If one is missing, likely the others weren’t recorded either—possibly due to incomplete clinical assessment.
- **{htn, dm, cad}:** All are comorbidities. If one is missing, the others are likely missing—possibly due to missing medical history.
- **{pcc, ba}:** Both are urine infection markers, often tested together.
- **{sod, pot}:** Both are electrolytes, always measured together in a metabolic panel.
- **{sg, al, su}:** All are urine test parameters, part of the same urinalysis.
- **{bu, sc}:** Both are kidney function markers, always ordered together.
- **{wc, rc}:** Both are blood cell counts, part of a CBC.
- **{hemo, pcv}:** Both are red cell measures, part of a CBC.
- **{hemo, pcv, wc, rc}:** All are CBC parameters.
- **{sg, al, su, rbc, pc}:** All are urinalysis parameters.

**_Medical Significance:_**  
These patterns reflect real-world clinical practice: tests are often ordered in panels, and if one is missing, the whole panel is likely missing. This is a sign of good data integrity.

---

#### **Moderate/Weak Positive Groups (MP/WP)**
- These show partial overlap, likely due to some tests being ordered together only in certain clinical scenarios (e.g., severe cases, follow-up visits).

### Way of Handling Missing Values:
- Low Missing (<5%)>:
  - Categorical
    - Filled with mode
  - Numerical
    - Filled with mean
- Too Many (>30%):
  - Categorical
    - Make separate category called missing
  - Numerical
    - None found
- Intermediate Amount (5-30%):
    - Mice Imputer using default imputer model

In [ ]:
low_missing_cats = ['pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']
for col in low_missing_cats:
    mode_val = df_clean[col].mode()[0]
    df_clean[col].fillna(mode_val, inplace=True)

In [ ]:
df_clean['rbc'] = df_clean['rbc'].fillna('missing')
df_clean['pc'] = df_clean['pc'].fillna('missing')

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler

def mice_impute_group(df, columns, estimator=None, max_iter=10, random_state=0):
    scaler = StandardScaler()
    scaled = scaler.fit_transform(df[columns])
    imputer = IterativeImputer(estimator=estimator, max_iter=max_iter, random_state=random_state)
    imputed = imputer.fit_transform(scaled)
    df[columns] = scaler.inverse_transform(imputed)
    return df

- Mice Imputer was chosen because it can impute columns together if they tend to be missing together as observed earlier through the missing value heatmap and correlation matrix based on patterns in the data, instead of imputing randomly
  - Relevant for medical data
  - Ensures integrity is maintained

In [ ]:
# Impute each group
df_clean = mice_impute_group(df_clean, ['sod', 'pot'])
df_clean = mice_impute_group(df_clean, ['sg', 'al', 'su'])
df_clean = mice_impute_group(df_clean, ['hemo', 'pcv', 'wc', 'rc'])
df_clean = mice_impute_group(df_clean, ['bu', 'sc', 'bgr'])  # bgr included for completeness

In [ ]:
df_clean['age'].fillna(df_clean['age'].mean(), inplace=True)
df_clean['bp'].fillna(df_clean['bp'].mean(), inplace=True)

In [ ]:
print(df_clean.isnull().sum())

### 'al' and 'su' Are Categorical Columns With Oridinal Categories From 0-5

In [ ]:
for col in ['al', 'su']:
    df_clean[col] = df_clean[col].round().astype(int)
    df_clean[col] = df_clean[col].clip(lower=0, upper=5)

### Done with all imputations neccessary

### Outlier Analysis and Removal

In [ ]:
numerical_cols = df_clean.select_dtypes(include=['float64', 'int64']).columns.tolist()
numerical_cols = [col for col in numerical_cols if col not in ['id']]  # Exclude id

plt.figure(figsize=(15, 2 * len(numerical_cols)))
for i, col in enumerate(numerical_cols, 1):
    plt.subplot(len(numerical_cols), 1, i)
    sns.boxplot(x=df_clean[col], color='skyblue')
    plt.title(f'Boxplot of {col}')
    plt.tight_layout()
plt.show()

#### Number of  Outliers in Numerical Columns

In [ ]:
outlier_summary = {}
for col in numerical_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df_clean[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)][col]
    outlier_summary[col] = {
        'lower_bound': lower_bound,
        'upper_bound': upper_bound,
        'num_outliers': outliers.count(),
        'outlier_values': outliers.values
    }
    print(f"{col}: {outliers.count()} outliers")

#### Outlier Summary to Take Further Action

In [ ]:
for col in outlier_summary:
    print(f"{col}: {outlier_summary[col]['num_outliers']} outliers")
    print(f"Lower Bound: {outlier_summary[col]['lower_bound']}")
    print(f"Upper Bound: {outlier_summary[col]['upper_bound']}")
    print(f"Outlier Values: {outlier_summary[col]['outlier_values']}")
    print("--------------------------------")


> **Outlier Handling Documentation**
>
> We checked all the medical data ranges for all these numerical columns for which box plots were made and decided to remove only those that were clinically impossible. Other outliers were possible, but as a result of severe disease. Since the other records provided valuable medical information, we decided to keep them.
> - **sod (Sodium):**  
>   - Removed rows where sod = 4.5 mEq/L (physiologically impossible).  
>   - Retained all other values, including extreme hyponatremia/hypernatremia, as these are possible in severe kidney disease.
>
> - **pot (Potassium):**  
>   - Removed rows where pot ≥ 39 mmol/L (clinically impossible).  
>   - Retained all other values, including severe hypo/hyperkalemia, as these are possible in advanced disease.
>
> - **pcv (Packed Cell Volume):**  
>   - Removed rows where pcv = 9% (clinically impossible).  
>   - Retained all other values, including severe anemia, as these are possible in advanced disease.
>
> - **All other columns:**  
>   - Retained all outlier values, as they are medically possible, though some are rare or dangerous. This preserves the clinical diversity of the dataset for robust modeling.


In [ ]:
# Count before removal
removed_sod = np.sum(np.isclose(df_clean['sod'], 4.5, atol=0.01))
removed_pot = np.sum(df_clean['pot'] >= 39)
removed_pcv = np.sum(np.isclose(df_clean['pcv'], 9.0, atol=0.01))

print(f"Rows to be removed for sod: {removed_sod}")
print(f"Rows to be removed for pot: {removed_pot}")
print(f"Rows to be removed for pcv: {removed_pcv}")

# Now remove the impossible values
df_clean = df_clean[~np.isclose(df_clean['sod'], 4.5, atol=0.01) | df_clean['sod'].isnull()]
df_clean = df_clean[(df_clean['pot'].isnull()) | (df_clean['pot'] < 39)]
df_clean = df_clean[~np.isclose(df_clean['pcv'], 9.0, atol=0.01)]

### Converting Number/Object to Category Wherever Needed

In [ ]:
# Convert to categorical with ordered categories
df_clean['al'] = pd.Categorical(df_clean['al'], categories=[0,1,2,3,4,5], ordered=True)
df_clean['su'] = pd.Categorical(df_clean['su'], categories=[0,1,2,3,4,5], ordered=True)

In [ ]:
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = df_clean[col].astype('category')

### Removing 'id' (Not useful for pattern finding)

In [ ]:
if 'id' in df_clean.columns:
    df_clean = df_clean.drop(columns=['id'])

# VISUALIZATIONS
## Univariate Analysis
1. Distribution Plots for Numerical Variables

In [ ]:
import os # Create a folder for plots if it doesn't exist
os.makedirs('plots', exist_ok=True)

In [ ]:
# List of numerical columns (excluding id if present)
numerical_cols = df_clean.select_dtypes(include=['float64', 'int64']).columns.tolist()
print(numerical_cols)
if 'id' in numerical_cols:
    numerical_cols.remove('id')

# Plot distributions
for col in numerical_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(df_clean[col], kde=True, color='royalblue', bins=30)
    plt.title(f'Distribution of {col}', fontsize=14)
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f'plots/hist_{col}.png')
    plt.show()
    # Interpretation: Look for skewness, multimodality, and outliers.

2. Distribution Again With Overlay of Normal Medical Ranges (Sometimes Separated by Gender)

In [ ]:
col='hemo'
plt.figure(figsize=(8, 4))
sns.histplot(df_clean[col], kde=True, color='royalblue', bins=30)
plt.axvspan(12.1, 15.1, color='green', alpha=0.2, label='Normal Female')
plt.axvspan(13.8, 17.2, color='orange', alpha=0.2, label='Normal Male')
plt.title(f'Distribution of {col}', fontsize=14)
plt.xlabel(col)
plt.ylabel('Count')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.legend()
plt.savefig(f'plots/hist_{col}_with_normal_ranges.png')
plt.show()

In [ ]:
col='bp'
plt.figure(figsize=(8, 4))
sns.histplot(df_clean[col], kde=True, color='royalblue', bins=30)
plt.axvspan(90, 120, color='green', alpha=0.2, label='Normal BP')
plt.title(f'Distribution of {col}', fontsize=14)
plt.xlabel(col)
plt.ylabel('Count')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(f'plots/hist_{col}_with_normal_ranges.png')
plt.show()

In [ ]:
col='sg'
plt.figure(figsize=(8, 4))
sns.histplot(df_clean[col], kde=True, color='royalblue', bins=30)
plt.axvspan(1.005, 1.030, color='green', alpha=0.2, label='Normal SG')
plt.title(f'Distribution of {col}', fontsize=14)
plt.xlabel(col)
plt.ylabel('Count')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.legend()
plt.savefig(f'plots/hist_{col}_with_normal_ranges.png')
plt.show()

In [ ]:
col='bgr'
plt.figure(figsize=(8, 4))
sns.histplot(df_clean[col], kde=True, color='royalblue', bins=30)
plt.axvspan(60,110, color='green', alpha=0.2, label=f'Normal {col.upper()}')
plt.title(f'Distribution of {col}', fontsize=14)
plt.xlabel(col)
plt.ylabel('Count')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.legend()
plt.savefig(f'plots/hist_{col}_with_normal_ranges.png')
plt.show()

In [ ]:
col='bu'
plt.figure(figsize=(8, 4))
sns.histplot(df_clean[col], kde=True, color='royalblue', bins=30)
plt.axvspan(10,50, color='green', alpha=0.2, label=f'Normal {col.upper()}')
plt.title(f'Distribution of {col}', fontsize=14)
plt.xlabel(col)
plt.ylabel('Count')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.legend()
plt.savefig(f'plots/hist_{col}_with_normal_ranges.png')
plt.show()

In [ ]:
col='sc'
plt.figure(figsize=(8, 4))
sns.histplot(df_clean[col], kde=True, color='royalblue', bins=30)
plt.axvspan(0.6,1.3, color='green', alpha=0.2, label=f'Normal {col.upper()}')
plt.title(f'Distribution of {col}', fontsize=14)
plt.xlabel(col)
plt.ylabel('Count')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.legend()
plt.savefig(f'plots/hist_{col}_with_normal_ranges.png')
plt.show()

In [ ]:
col='sod'
plt.figure(figsize=(8, 4))
sns.histplot(df_clean[col], kde=True, color='royalblue', bins=30)
plt.axvspan(135,145, color='green', alpha=0.2, label=f'Normal {col.upper()}')
plt.title(f'Distribution of {col}', fontsize=14)
plt.xlabel(col)
plt.ylabel('Count')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.legend()
plt.savefig(f'plots/hist_{col}_with_normal_ranges.png')
plt.show()

In [ ]:
col='pot'
plt.figure(figsize=(8, 4))
sns.histplot(df_clean[col], kde=True, color='royalblue', bins=30)
plt.axvspan(3.5,5.0, color='green', alpha=0.2, label=f'Normal {col.upper()}')
plt.title(f'Distribution of {col}', fontsize=14)
plt.xlabel(col)
plt.ylabel('Count')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.legend()
plt.savefig(f'plots/hist_{col}_with_normal_ranges.png')
plt.show()

In [ ]:
col='pcv'
plt.figure(figsize=(8, 4))
sns.histplot(df_clean[col], kde=True, color='royalblue', bins=30)
plt.axvspan(36,54, color='green', alpha=0.2, label=f'Normal {col.upper()}')
plt.title(f'Distribution of {col}', fontsize=14)
plt.xlabel(col)
plt.ylabel('Count')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.legend()
plt.savefig(f'plots/hist_{col}_with_normal_ranges.png')
plt.show()

In [ ]:
col='wc'
plt.figure(figsize=(8, 4))
sns.histplot(df_clean[col], kde=True, color='royalblue', bins=30)
plt.axvspan(4500,11000, color='green', alpha=0.2, label=f'Normal {col.upper()}')
plt.title(f'Distribution of {col}', fontsize=14)
plt.xlabel(col)
plt.ylabel('Count')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.legend()
plt.savefig(f'plots/hist_{col}_with_normal_ranges.png')
plt.show()

In [ ]:
col='rc'
plt.figure(figsize=(8, 4))
sns.histplot(df_clean[col], kde=True, color='royalblue', bins=30)
plt.axvspan(4.1,5.7, color='green', alpha=0.2, label=f'Normal {col.upper()}')
plt.title(f'Distribution of {col}', fontsize=14)
plt.xlabel(col)
plt.ylabel('Count')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.legend()
plt.savefig(f'plots/hist_{col}_with_normal_ranges.png')
plt.show()

3. Distribution With CKD Status Hue

In [ ]:
numerical_cols = df_clean.select_dtypes(include=['float64', 'int64']).columns.tolist()

for col in numerical_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(data=df_clean, x=f'{col}', hue='classification', kde=True)
    plt.title(f'{col.upper()} by CKD Status')
    plt.savefig(f'plots/hist_{col}_by_ckd.png')
    plt.show()

4. Numerical Columns With Hue for CKD Status **Proportion**

In [ ]:
numerical_cols=df_clean.select_dtypes(include=['float64', 'int64']).columns.tolist()
for col in numerical_cols:
    plt.figure(figsize=(8, 5))
    sns.histplot(data=df_clean, x=f'{col}', hue='classification', bins=20, kde=True, palette='Set1', multiple='stack')
    plt.title(f'{col.title()} Distribution by CKD Status')
    plt.xlabel(f'{col.title()}')
    plt.ylabel('Count')
    plt.legend(title='CKD Status', labels=df_clean['classification'].unique())
    plt.tight_layout()
    plt.savefig(f'plots/hist_{col}_by_ckd.png')
    plt.show()

In [ ]:
df_clean['classification'] = df_clean['classification'].astype(str)

#### Analysis of Distributions

We observed the following patterns across various features:

##### **Age**
- **Skewness:** Slightly right-skewed  
- **Modality:** Unimodal (1 peak)  
- **Key Observations:** Most individuals are above mid-40s (45) but under 70.  

##### **Blood Pressure (bp)**
- **Skewness:** Highly left-skewed  
- **Modality:** Unimodal (tiny secondary bumps at the top)  
- **Key Observations:** The data is concentrated within fixed ranges, with sparse values otherwise. Most values fall within **90–120 mmHg (systolic)**.  

##### **Specific Gravity (sg)**
- **Skewness:** Not skewed  
- **Modality:** Unimodal (though curves are wavy with relatively even short hills)  
- **Key Observations:** KDE is spread in a **constant wavy pattern**, initially low, picking up from **1.0075**. Most values fall within **1.005 to 1.030**.  

## **Blood Glucose Random (bgr)**
- **Skewness:** Highly left-skewed  
- **Modality:** Unimodal  
- **Key Observations:** Most values range **between 50–160 mg/dL**, whereas typical healthy levels are **60–110 mg/dL**.  

#### **Blood Urea (bu)**
- **Skewness:** Highly left-skewed  
- **Modality:** Unimodal  
- **Key Observations:** Most values range between **0–80 mg/dL**, while normal levels typically fall **between 10–50 mg/dL**.  

#### **Serum Creatinine (sc)**
- **Skewness:** Highly left-skewed  
- **Modality:** Unimodal  
- **Key Observations:** Most values lie **between 0–4 mg/dL**, while normal adult levels are **0.6–1.3 mg/dL**.  

#### **Sodium (sod)**
- **Skewness:** Highly right-skewed  
- **Modality:** Unimodal  
- **Key Observations:** Most values fall **between 136–142 mEq/L**, aligning with typical sodium levels **(135–145 mEq/L)**.  

#### **Potassium (pot)**
- **Skewness:** No significant skew  
- **Modality:** Unimodal (increasing initially, steady, then a peak mid-range, followed by a decrease)  
- **Key Observations:** Most values range **between 3.3–5 mmol/L**, consistent with normal blood potassium levels **(3.5–5.0 mmol/L)**.  

#### **Hemoglobin (hemo)**
- **Skewness:** No significant skew (almost normal but short and stout distribution)  
- **Modality:** Unimodal  
- **Key Observations:** A substantial peak is seen **around 12 g/dL**, though values are well distributed.   
  - **Does this suggest a female-dominated sample?** Normal **female hemoglobin levels are 12.1–15.1 g/dL**, while **male levels range from 13.8–17.2 g/dL**.  
  - **Without a gender column, direct validation isn't possible**, but overlap between the ranges suggests potential diversity in the dataset.  

#### **Packed Cell Volume (pcv)**
- **Skewness:** Slightly right-skewed (almost normal but short and stout)  
- **Modality:** Unimodal  
- **Key Observations:** Most values cluster **around 40%**, increasing gradually before peaking.  
  - Normal adult PCV levels fall within **36–54%**.  

#### **White Blood Cell Count (wc)**
- **Skewness:** Highly left-skewed  
- **Modality:** Unimodal  
- **Key Observations:** Most values fall **between 6000–11000/μL**, which aligns with typical WBC count **(4,500–11,000/μL)**.  

#### **Red Blood Cell Count (rc)**
- **Skewness:** No significant skew (almost normal but rapid decrease after the center)  
- **Modality:** Unimodal  
- **Key Observations:** Large peak **centered at around 4.5 million/mcL**.  
  - Typical RBC levels range **between 4.1–5.7 million/mcL**.  


### Count Plots For Categorical Columns

In [ ]:
# List of categorical columns
cat_cols = df_clean.select_dtypes(include='category').columns.tolist()

for col in cat_cols:
    plt.figure(figsize=(7, 4))
    sns.countplot(x=col, data=df_clean, palette='Set2')
    plt.title(f'Count Plot of {col}', fontsize=14)
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f'plots/count_{col}.png')
    plt.show()
    # Interpretation: See class balance, rare categories, and missing category if present.

## Bivariate Analysis
1. Correlation Heatmap for Numerical Variables (Before Feature Engineered Columns Were Added)

In [ ]:
plt.figure(figsize=(14, 10))
corr = df_clean[numerical_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of Numerical Features', fontsize=16)
plt.tight_layout()
plt.savefig('plots/corr_heatmap.png')
plt.show()
# Interpretation: Look for strong positive/negative correlations, which may indicate redundancy or important relationships.

## **Interpretation of Column Correlations in Chronic Kidney Disease (CKD) Dataset**

The correlation table provides **valuable insights** into how different features interact, indicating potential **biological dependencies** or **redundancy** in variables. Below is a breakdown of each column’s correlations, with reasoning behind them.

---

### **1. Age:** Low correlations  
- Age shows no strong correlation with other features, which suggests **it does not directly affect immediate kidney function markers** in this dataset.
- **Possible Explanation:** Age is a **background factor** that affects disease progression over time, but kidney disease is more **driven by biochemical markers** than raw age alone.

---

### **2. Blood Pressure (bp): Low correlations**  
- Max correlation: **0.29**
- **Possible Explanation:** While **high blood pressure** is a major **risk factor** for kidney disease, this dataset might not capture a **direct relationship** between BP and renal biomarkers due to variability in hypertension medication or individual patient responses.

---

### **3. Specific Gravity (sg):**  
- **Positive correlations:**  
  - **0.49 (rc), 0.52 (pcv), 0.53 (hemo), 0.37 (sod)**  
- **Negative correlation:**  
  - **-0.32 (bgr)**  
- **Biological Interpretation:**  
  - SG measures **urine concentration**; it **correlates positively with blood health markers** (**RBC, PCV, Hemoglobin**) because higher SG suggests **better kidney filtration**.  
  - Negative correlation with **blood glucose (bgr)** indicates that **higher SG may be associated with lower glucose levels**, possibly reflecting **normal kidney function** in regulating glucose excretion.

---

### **4. Blood Glucose (bgr):**  
- **Negative correlation:**  
  - **-0.32 (sg)**  
- **Biological Interpretation:**  
  - Lower specific gravity **may be linked to increased glucose levels in urine**, as seen in **diabetic nephropathy**.

---

### **5. Blood Urea (bu):**  
- **Negative correlations:**  
  - **-0.51 (rc), -0.55 (pcv), -0.55 (hemo), -0.40 (sod)**  
- **Positive correlation:**  
  - **0.63 (sc)**  
- **Biological Interpretation:**  
  - **Higher blood urea (BU) is linked to kidney dysfunction** → The **negative correlation with RBC, hemoglobin, PCV, and sodium** suggests that **poor kidney function leads to anemia and electrolyte imbalances**.  
  - **Strong correlation with serum creatinine (SC)** confirms **BU and SC rise together in cases of kidney failure**.

---

### **6. Serum Creatinine (sc):**  
- **Negative correlations:**  
  - **-0.41 (rc), -0.44 (pcv), -0.44 (hemo), -0.38 (sod)**  
- **Positive correlation:**  
  - **0.63 (bu)**  
- **Biological Interpretation:**  
  - **SC is a primary marker of kidney filtration efficiency**; higher SC indicates **poor kidney function**, leading to **reduced RBC, PCV, and hemoglobin (signs of anemia)**.  
  - **Very similar correlation values with BU**, confirming both SC and BU **rise together as kidney function declines**.

---

### **7. Sodium (sod):**  
- **Positive correlations:**  
  - **0.41 (rc), 0.47 (pcv), 0.47 (hemo), 0.37 (sg)**  
- **Negative correlations:**  
  - **-0.38 (sc), -0.40 (bu)**  
- **Biological Interpretation:**  
  - Sodium plays a **major role in fluid balance and kidney function**.  
  - Higher sodium correlates with **higher RBC, PCV, and hemoglobin**, suggesting that **normal sodium levels maintain blood health**.  
  - **Negative correlation with SC and BU** indicates that **kidney failure causes sodium imbalance**.

---

### **8. Hemoglobin (hemo) & Packed Cell Volume (pcv):**  
- **Positive correlations:**  
  - **0.84 (rc), 0.9 (pcv), 0.47 (sod), 0.53 (sg)**  
- **Negative correlations:**  
  - **-0.44 (sc), -0.55 (bu)**  
- **Biological Interpretation:**  
  - **Hemo and PCV show identical correlation behavior**, reinforcing the fact that **PCV and hemoglobin measure blood oxygen-carrying capacity**.  
  - Their **high correlation with RBC (0.84) confirms anemia is a major outcome of kidney disease**.  
  - **Negative correlation with BU and SC** suggests that **kidney dysfunction directly leads to anemia**.

---

### **9. Red Blood Cells (rc):**  
- **Positive correlations:**  
  - **0.83 (pcv), 0.84 (hemo), 0.41 (sod), 0.49 (sg)**  
- **Negative correlations:**  
  - **-0.41 (sc), -0.51 (bu)**  
- **Biological Interpretation:**  
  - RBC levels **depend on kidney function**, since kidneys produce **erythropoietin**, a hormone regulating RBC production.  
  - **Declining kidney function (high SC, BU) reduces RBC count**, leading to **anemia**.  
  - **Positive correlation with PCV and hemoglobin** further proves **RBC is tightly linked to anemia indicators**.

---

### **10. Weak Correlations (wc, pot, bp):**  
- **Potassium (pot), white blood cell count (wc), and blood pressure (bp) show weak correlations** across other features.  
- **Possible Explanation:**  
  - **Potassium regulation is complex**, influenced by **diet, kidney function, and adrenal hormones**.  
  - **WBC count does not strongly correlate with kidney markers** unless an infection or inflammatory condition is present.  
  - **BP correlations might be weak due to varied individual responses to hypertension treatment**.

---

### **Key Insights from the Correlation Analysis**
- ✔ **SC and BU are closely linked to kidney failure** (strong correlation: **0.63**).
- ✔ **Hemo, PCV, and RBC behave similarly** (highly correlated with each other).
- ✔ **SG correlates positively with blood health markers**, reinforcing its role in **kidney filtration efficiency**.
- ✔ **Sodium imbalances occur as kidney function declines**, affecting **blood oxygen levels**.
- ✔ **BP, Potassium, and WBC do not show strong relationships with kidney disease markers**.

---

2. Pair Plots for Key Kidney Function Markers

In [ ]:
# Example: bu, sc, sod, pot, hemo
key_markers = [col for col in ['bu', 'sc', 'sod', 'pot', 'hemo'] if col in df_clean.columns]
sns.pairplot(df_clean[key_markers], diag_kind='kde', corner=True)
plt.suptitle('Pair Plot of Key Kidney Function Markers', y=1.02, fontsize=16)
plt.savefig('plots/pairplot_kidney_markers.png')
plt.show()
# Interpretation: Visualize relationships and clusters among key lab values.

3.  Feature Relationships with Medical Context

- htn (Hypertension) and bp (Blood Pressure)

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='htn', y='bp', data=df_clean, palette='Set1')
plt.title('Blood Pressure by Hypertension Status', fontsize=14)
plt.xlabel('Hypertension')
plt.ylabel('Blood Pressure (mmHg)')
plt.tight_layout()
plt.savefig('plots/bp_by_htn.png')
plt.show()
# Interpretation: Expect higher BP in 'yes' htn group.

- dm (Diabetes Mellitus) and bgr (Blood Glucose Random)

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='dm', y='bgr', data=df_clean, palette='Set1')
plt.title('Blood Glucose by Diabetes Status', fontsize=14)
plt.xlabel('Diabetes Mellitus')
plt.ylabel('Blood Glucose (mg/dL)')
plt.tight_layout()
plt.savefig('plots/bgr_by_dm.png')
plt.show()
# Interpretation: Expect higher bgr in 'yes' dm group.

- ane (Anemia) with hemo, pcv, rc

In [ ]:
for col in ['hemo', 'pcv', 'rc']:
    if col in df_clean.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x='ane', y=col, data=df_clean, palette='Set1')
        plt.title(f'{col} by Anemia Status', fontsize=14)
        plt.xlabel('Anemia')
        plt.ylabel(col)
        plt.tight_layout()
        plt.savefig(f'plots/{col}_by_ane.png')
        plt.show()
# Interpretation: Expect lower values in 'yes' ane group.

- pe (Pedal Edema) with sod and pot

In [ ]:
for col in ['sod', 'pot']:
    if col in df_clean.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x='pe', y=col, data=df_clean, palette='Set1')
        plt.title(f'{col} by Pedal Edema Status', fontsize=14)
        plt.xlabel('Pedal Edema')
        plt.ylabel(col)
        plt.tight_layout()
        plt.savefig(f'plots/{col}_by_pe.png')
        plt.show()
# Interpretation: Look for electrolyte imbalances in 'yes' pe group.

- appet (Appetite) with bu and sc

In [ ]:
for col in ['bu', 'sc']:
    if col in df_clean.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x='appet', y=col, data=df_clean, palette='Set1')
        plt.title(f'{col} by Appetite Status', fontsize=14)
        plt.xlabel('Appetite')
        plt.ylabel(col)
        plt.tight_layout()
        plt.savefig(f'plots/{col}_by_appet.png')
        plt.show()
# Interpretation: Poor appetite may be associated with higher bu/sc (uremia).

# Target Variable Analysis
## Distribution of CKD Status

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='classification', data=df_clean, palette='Set2')
plt.title('Distribution of CKD Status', fontsize=14)
plt.xlabel('CKD Status')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('plots/ckd_status.png')
plt.show()
# Interpretation: Check for class imbalance.

## Feature Distributions by CKD Status

In [ ]:
for col in numerical_cols:
    plt.figure(figsize=(8, 4))
    sns.kdeplot(data=df_clean, x=col, hue='classification', fill=True, common_norm=False, palette='Set1')
    plt.title(f'{col} Distribution by CKD Status', fontsize=14)
    plt.xlabel(col)
    plt.ylabel('Density')
    plt.tight_layout()
    plt.savefig(f'plots/{col}_by_ckd.png')
    plt.show()
# Interpretation: See how each feature separates CKD vs. notCKD.

# In Depth Analysis Based on Correlations:

a. Kidney Function and Anemia
- SC, BU vs. Hemo, PCV, RC:
- These relationships are central to CKD progression (as kidney function declines, anemia worsens).

In [ ]:
# Scatterplots with regression lines
sns.lmplot(x='sc', y='hemo', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Serum Creatinine vs. Hemoglobin by CKD Status')
plt.show()

sns.lmplot(x='bu', y='hemo', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Blood Urea vs. Hemoglobin by CKD Status')
plt.show()

sns.lmplot(x='sc', y='pcv', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Serum Creatinine vs. Packed Cell Volume by CKD Status')
plt.show()

sns.lmplot(x='bu', y='pcv', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Blood Urea vs. Packed Cell Volume by CKD Status')
plt.show()

sns.lmplot(x='sc', y='rc', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Serum Creatinine vs. RBC by CKD Status')
plt.show()

sns.lmplot(x='bu', y='rc', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Blood Urea vs. RBC by CKD Status')
plt.show()

sns.lmplot(x='sc', y='sod', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Serum Creatinine vs. Sodium by CKD Status')
plt.show()

sns.lmplot(x='bu', y='sod', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Blood Urea vs. Sodium by CKD Status')
plt.show()

- Notice negative slopes and separation by CKD status.


b. Sodium and Blood Health
- Sodium vs. Hemo/PCV/RC:
- Visualize how sodium imbalances relate to anemia.

In [ ]:
sns.lmplot(x='sod', y='hemo', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Sodium vs. Hemoglobin by CKD Status')
plt.show()

- Notice the border below which you certainly have Kidney Disease (indicated by lower hemo)

c. Specific Gravity and Blood Markers
- SG vs. Hemo/PCV/RC:
- Higher SG may indicate better kidney function and blood health.

In [ ]:
sns.lmplot(x='sg', y='hemo', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Specific Gravity vs. Hemoglobin by CKD Status')
plt.show()

- Imagine a 2X2 grid covering the entire plot.
- **Right Diagonal**
- High hemo and sg =>NotCKD (max occurences)
- Low hemo and sg=> CKD
- **Left Diagonal**
- High hemo and low sg=> CKD
- High sg and low hemo=> CKD

- _Hemo and sg together are very good indicators of CKD, because they clearly separate the different cases as is visible if we imagine a 2x2 grid [may not be as good independently as shown in the hue based histogram]_


2. Multivariate Patterns: PairGrid/Pairplot with Hue
- Visualize clusters and separation between CKD and non-CKD.

In [ ]:
key_vars = ['sc', 'bu', 'hemo', 'pcv', 'rc', 'sod']
sns.pairplot(df_clean[key_vars + ['classification']], hue='classification', diag_kind='kde', corner=True, plot_kws={'alpha':0.5})
plt.suptitle('Pairwise Relationships of Key CKD Markers', y=1.02)
plt.show()

- Take one of hemo, pcv, rc

3. Explore Weak/Unexpected Correlations
- Boxplots of Potassium, WBC, BP by CKD status and by other comorbidities (htn, dm, ane).
- This can reveal if these features are more important in certain subgroups.

In [ ]:
for col in ['pot', 'wc', 'bp']:
    plt.figure(figsize=(8,5))
    sns.boxplot(x='classification', y=col, data=df_clean, palette='Set2')
    plt.title(f'{col} by CKD Status')
    plt.show()
    # By comorbidity
    for comorb in ['htn', 'dm', 'ane']:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=comorb, y=col, hue='classification', data=df_clean, palette='Set3')
        plt.title(f'{col} by {comorb}')
        plt.show()

- wb,pot,bp values are more variable or extreme in their graphs with 'classification' when CKD is true.
- Notice how, having any of the comorbid conditions [ane, htn, dm] confirms CKD, but not having them doesn't confirm not having the same.
- This means if one doesn't have these comorbid conditions, they will have to do other tests to confirm for sure.
- But, if they do, there is a high chance they have kidney disease.

4. Interaction Plots
- Interaction between two features and CKD status.
- Example: How does the relationship between SC/BU [Kidney Functioning Indicators] and Hemo differ by CKD status?

In [ ]:
df_clean['sc_bin'] = pd.cut(df_clean['sc'], bins=5)
df_clean['hemo_bin'] = pd.cut(df_clean['hemo'], bins=5)
pivot = df_clean.pivot_table(index='sc_bin', columns='hemo_bin', values='classification', aggfunc=lambda x: (x=='ckd').mean())
plt.figure(figsize=(8,6))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap='coolwarm')
plt.title('CKD Rate by SC and Hemoglobin Bins')
plt.xlabel('Hemoglobin Bin')
plt.ylabel('Serum Creatinine Bin')
plt.show()

In [ ]:
df_clean['bu_bin'] = pd.cut(df_clean['bu'], bins=5)
df_clean['hemo_bin'] = pd.cut(df_clean['hemo'], bins=5)
pivot = df_clean.pivot_table(index='bu_bin', columns='hemo_bin', values='classification', aggfunc=lambda x: (x=='ckd').mean())
plt.figure(figsize=(8,6))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap='coolwarm')
plt.title('CKD Rate by BU and Hemoglobin Bins')
plt.xlabel('Hemoglobin Bin')
plt.ylabel('Blood Urea Bin')
plt.show()

## Feature Engineering for CKD
1. eGFR Calculation (Estimated Glomerular Filtration Rate)
- eGFR is a standard kidney function score, calculated from serum creatinine, age, sex, and sometimes race.
- Since you don’t have sex/race, use the simplified MDRD formula (for adults):
    - eGFR=186×(sc)^(−1.154)×(age)^(−0.203)

In [ ]:
# Avoid division by zero or negative values
df_clean['eGFR'] = 186 * (df_clean['sc'].clip(lower=0.01))**(-1.154) * (df_clean['age'].clip(lower=1))**(-0.203)

In [ ]:
sns.lmplot(x='sc', y='eGFR', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Serum Creatinine vs. eGFR by CKD Status')
plt.show()

sns.lmplot(x='bu', y='eGFR', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Blood Urea vs. eGFR by CKD Status')
plt.show()

2. Comorb_Score (Level of Comorbid Conditions in One's Body)
- 0 (none) to 3 (all conditions)
- Each "yes" = 1 point.

In [ ]:
df_clean['comorb_score'] = (
    (df_clean['htn'] == 'yes').astype(int) +
    (df_clean['dm'] == 'yes').astype(int) +
    (df_clean['cad'] == 'yes').astype(int)
)

#### Checking trend formed with respect to sc and bu (Expecting positive slope)
- Higher sc/bu => Higher cormorbid condition level

In [ ]:
sns.lmplot(x='sc', y='comorb_score', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Serum Creatinine vs. comorb_score by CKD Status')
plt.show()

sns.lmplot(x='bu', y='comorb_score', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Blood Urea vs. comorb_score by CKD Status')
plt.show()

3. Anemia Severity Score
- Combine hemo, pcv, rc into a z-score-based severity index.
- Interpretation: Higher value = more severe anemia.

In [ ]:
from scipy.stats import zscore

# Calculate z-scores (lower = more severe anemia)
df_clean['anemia_severity'] = -(
    zscore(df_clean['hemo']) +
    zscore(df_clean['pcv']) +
    zscore(df_clean['rc'])
)

#### Checking trend formed with respect to sc and bu (Expecting positive slope)
- Higher sc/bu => Higher anemia severity

In [ ]:
sns.lmplot(x='sc', y='anemia_severity', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Serum Creatinine vs. anemia_severity by CKD Status')
plt.show()

sns.lmplot(x='bu', y='anemia_severity', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Blood Urea vs. anemia_severity by CKD Status')
plt.show()

4. Kidney Function Score
- Interpretation: Higher value = worse kidney function.

In [ ]:
df_clean['kidney_func_score'] = (
    zscore(df_clean['bu']) +
    zscore(df_clean['sc']) -
    zscore(df_clean['sod'])
)

#### Checking trend formed with respect to sc and bu (Expecting positive slope)
- Higher sc/bu => Higher kidney function score

In [ ]:
sns.lmplot(x='sc', y='kidney_func_score', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Serum Creatinine vs. kidney_func_score by CKD Status')
plt.show()

sns.lmplot(x='bu', y='kidney_func_score', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Blood Urea vs. kidney_func_score by CKD Status')
plt.show()

5. Symptom Severity Score
- Combine appet, pe, ane (each "poor"/"yes" = 1 point).
- Interpretation: Higher score = more severe symptoms.

In [ ]:

df_clean['symptom_severity'] = (
    (df_clean['appet'] == 'poor').astype(int) +
    (df_clean['pe'] == 'yes').astype(int) +
    (df_clean['ane'] == 'yes').astype(int)
)

#### Checking trend formed with respect to sc and bu (Expecting positive slope)
- Higher sc/bu => Higher symptom severity level

In [ ]:
sns.lmplot(x='sc', y='symptom_severity', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Serum Creatinine vs. symptom_severity by CKD Status')
plt.show()

sns.lmplot(x='bu', y='symptom_severity', hue='classification', data=df_clean, aspect=1.2, height=5, scatter_kws={'alpha':0.5})
plt.title('Blood Urea vs. symptom_severity by CKD Status')
plt.show()

### New Set of Numerical Columns

In [ ]:
numerical_cols=df_clean.select_dtypes(include=['float64', 'int64']).columns.tolist()

In [ ]:
print(numerical_cols)

### Correlation Heatmap for all Variables (After Feature Engineered Columns Have Been Added)

In [ ]:
# Identify numerical and categorical columns
numerical_cols = df_clean.select_dtypes(include=['number']).columns
categorical_cols = df_clean.select_dtypes(include=['category']).columns

# Apply One-Hot Encoding to categorical columns
df_encoded = pd.get_dummies(df_clean[categorical_cols], drop_first=True)

# Combine numerical and encoded categorical columns
df_new = pd.concat([df_clean[numerical_cols], df_encoded], axis=1)

# Compute correlation matrix
corr_matrix = df_new.corr()

# Plot correlation matrix
plt.figure(figsize=(30, 26))
sns.heatmap(corr_matrix, cmap="coolwarm", annot=True, fmt=".2f")
plt.title("Correlation Matrix of Numerical & One-Hot Encoded Categorical Columns")
plt.show()


### **1. eGFR**  
- **Positive correlations:**  
  - **0.49 (hemo), 0.48 (pcv), 0.51 (classification_not_ckd)**  
- **Negative correlation:**  
  - **-0.49 (bu), -0.45 (sc), -0.49 (comorb_score), -0.49 (anemia_severity), -0.53 (kidney_func_score), -0.47 (htn_yes)**  
- **Biological Interpretation:**  
  - eGFR decreases with worsening kidney function, which is reflected by higher urea and creatinine. Poor kidney function leads to anemia (low hemoglobin, PCV) and coexists with comorbidities and symptoms. Better eGFR is protective and associated with being classified as not having CKD.


### **2. Comorbidity Score**  
- **Positive correlations:**  
  - **0.49 (bgr), 0.62 (anemia_severity), 0.46 (kidney_func_score), 0.47 (symptom_severity), 0.87 (htn_yes), 0.85 (dm_yes), 0.57 (cad_yes)**  
- **Negative correlation:**  
  - **-0.59 (hemo), -0.60 (pcv), -0.58 (rc), -0.49 (eGFR), -0.63 (classification_notckd))**  
- **Biological Interpretation:**  
  - Higher comorbidity scores indicate the presence of diseases like diabetes, hypertension, and anemia. These are linked with worsening kidney and blood parameters. Comorbidities also increase symptom burden and are strongly associated with CKD classification.


### **3. Anemia Severity**  
- **Positive correlations:**  
  - **0.57 (bu), 0.45 (sc), 0.47 (sod), 0.62 (comorb_score), 0.62 (kidney_func_score), 0.61 (symptom_severity), 0.61 (htn_yes), 0.48 (dm_yes), 0.55 (ane_yes), 0.58 (hemo_bin)**  
- **Negative correlation:**  
  - **-0.54 (sg), -0.96 (hemo), -0.96 (pcv), -0.93 (rc), -0.49 (eGFR), -0.47 (pc_normal), -0.46 (rbc_normal), -0.74 (classification_notckd), -0.54 (hemo_bin))**  
- **Biological Interpretation:**  
  - Anemia severity increases with poor kidney function, as kidneys produce less erythropoietin. Hemoglobin, PCV, and RBC are directly affected. Electrolyte imbalances and comorbidities further exacerbate anemia. CKD patients commonly present with higher anemia severity.

  


### **4. Kidney Function Score**  
- **Positive correlations:**  
  - **0.84 (bu), 0.83 (sc), 0.46 (comorb_score), 0.62 (anemia_severity), 0.51 (symptom_severity), 0.46 (htn_yes), 0.46 (ane_yes), 0.62 (sc_bin), 0.46 (bu_bin)**  
- **Negative correlation:**  
  - **-0.74 (sod), -0.60 (hemo), -0.61 (pcv), -0.55 (rc), -0.53 (eGFR), -0.51 (classification_no_ckd)**  
- **Biological Interpretation:**  
  - The kidney function score reflects deteriorating kidney health. As it worsens, markers like urea and creatinine rise, while sodium and hemoglobin drop. Higher scores are linked with more comorbidities, anemia, and CKD-related symptoms.


### **5. Symptom Severity**  
- **Positive correlations:**  
  - **0.48 (bu), 0.47 (comorb_score), 0.61 (anemia_severity), 0.51 (kidney_func_score), 0.49 (htn_yes), 0.78 (appet_poor), 0.75 (pe_yes), 0.64 (ane_yes)**  
- **Negative correlation:**  
  - **-0.59 (hemo), -0.59 (pcv), -0.56 (rc), -0.50 (classification_not_ckd)**  
- **Biological Interpretation:**  
  - As kidney disease progresses, symptoms like fatigue, appetite loss, and edema increase. These symptoms correlate with anemia, poor kidney function, and comorbid conditions. CKD patients consistently report higher symptom severity.

In [ ]:
!pip install statsmodels

- Scaling
- Normalisation
- Take different subsets of dataset and test on all models to find best performing
- Train test split
- Model selection from all options
- Metrics

### Distribution of New Numerical Columns

In [ ]:
for col in ['eGFR', 'anemia_severity', 'kidney_func_score','symptom_severity','comorb_score']:
    plt.figure(figsize=(8, 4))
    sns.histplot(df_clean[col], kde=True, color='royalblue', bins=30)
    plt.title(f'Distribution of {col}', fontsize=14)
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f'plots/hist_{col}.png')
    plt.show()
    # Interpretation: Look for skewness, multimodality, and outliers.

#### Analysis of Distributions

We observed the following patterns across various features:

>Analysis:
>##### **eGFR**
- **Skewness:** Slightly right-skewed  
- **Modality:** Unimodal (1 peak)  
- **Key Observations:** Most individuals are above mid-40s (45) but under 70.  

>##### **anemia_severity**
- **Skewness:** Slightly right-skewed  
- **Modality:** Bimodal (peaks near -2 and 0)
- **Key Observations:** Sharp spike at 0 possibly indicating a subgroup; most values range between -3 and 4.

>##### **kidney_func_score**
- **Skewness:** Slightly right-skewed  
- **Modality:** Unimodal (1 peak around -1.5)
- **Key Observations:** Majority of values lie between -3 and 2, with a long tail extending toward higher scores.

>##### **symptom_severity**
- **Skewness:** Right-skewed  
- **Modality:** Multimodal (distinct peaks at 0, 1, 2, and 3)
- **Key Observations:** Majority of patients report no symptoms (0), followed by a steep drop for each increasing severity level. Values are discrete and clustered.


>##### **comorb_score**
- **Skewness:** Right-skewed  
- **Modality:** Multimodal (distinct bars at 0, 1, 2, 3)
- **Key Observations:** Most patients have zero comorbidities, but there's a clear step-wise pattern showing others with 1, 2, or 3 conditions. The score appears categorical or count-based.

### Ensuring the datatypes and non-null values are fine before moving on to train-test-split, scaling and transforms

In [ ]:
df_clean.info()

In [ ]:
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = df_clean[col].astype('category')
    print('One Only.')

In [ ]:
df_clean.info()

### Transformation and Scaling Plan

> Data Transformation Steps:
> - Log-transformed highly skewed variables: bgr, bu, sc, wc, eGFR.
> - Standard scaled all numerical features.
> - One-hot encoded all categorical variables.
> - All transformations were fit on the training set and applied to the test set to prevent data leakage.

#### Ensuring None That Require Log Transforms Have 0 or -ve Values

In [ ]:
cols_to_check = ['bgr', 'bu', 'sc', 'wc', 'eGFR']

for col in cols_to_check:
    if col in df_clean.columns:
        num_zeros = (df_clean[col] == 0).sum()
        num_neg = (df_clean[col] < 0).sum()
        print(f"{col}: {num_zeros} zeros, {num_neg} negatives")
        print(f"Min value: {df_clean[col].min()}")
        print(f"Type: {df_clean[col].dtype}\n")

In [ ]:
from sklearn.model_selection import train_test_split

# Exclude target and categorical columns from features
target = 'classification'
feature_cols = [col for col in df_clean.columns if col not in [target, 'sc_bin', 'hemo_bin', 'bu_bin']]  # Exclude binned cols and target

X = df_clean[feature_cols]
y = df_clean[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
print(X_train.columns.tolist())

In [ ]:
print(X_train.head())

In [ ]:
print(y)

In [ ]:
log_transform_cols = ['bgr', 'bu', 'sc', 'wc', 'eGFR']

for col in log_transform_cols:
    for df in [X_train, X_test]:
        df[col] = np.log(df[col])

In [ ]:
X_train.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

# Identify numerical columns (excluding categorical and binned)
num_cols = X_train.select_dtypes(include=['float64', 'int32']).columns.tolist()

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [ ]:
X_train.head()

In [ ]:
cat_cols = X_train.select_dtypes(include='category').columns.tolist()

X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

# Ensure columns match in train and test
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

print(le.classes_)  # To see which label is 0 and which is 1

y_train = pd.Series(y_train)
y_test = pd.Series(y_test)

In [ ]:
X_train.head()

In [ ]:
print(X_train.isnull().sum().sum(), X_test.isnull().sum().sum())

In [ ]:
print(X_train.shape, X_test.shape)

In [ ]:
print(df_clean.columns.tolist())

In [ ]:
#most imporant features
feature_cols_imp1 = ['anemia_severity','hemo','sg','comorb_score','eGFR','rbc','sc','dm','htn','bgr','symptom_severity']

X_imp1 = df_clean[feature_cols_imp1]
y_imp1 = df_clean[target]

X_train_imp1, X_test_imp1, y_train_imp1, y_test_imp1 = train_test_split(X_imp1, y_imp1, test_size=0.2, random_state=42, stratify=y_imp1)

In [ ]:
y_train_imp1.value_counts()

In [ ]:
len(feature_cols_imp1)

In [ ]:
log_transform_cols_imp1 = ['bgr','sc', 'eGFR']

for col in log_transform_cols_imp1:
    for df in [X_train_imp1, X_test_imp1]:
        df[col] = np.log(df[col])

In [ ]:
from sklearn.preprocessing import StandardScaler

# Identify numerical columns (excluding categorical and binned)
num_cols_imp1 = X_train_imp1.select_dtypes(include=['float64', 'int32']).columns.tolist()

scaler_imp1 = StandardScaler()
X_train_imp1[num_cols_imp1] = scaler_imp1.fit_transform(X_train_imp1[num_cols_imp1])
X_test_imp1[num_cols_imp1] = scaler_imp1.transform(X_test_imp1[num_cols_imp1])

In [ ]:
cat_cols_imp1 = X_train_imp1.select_dtypes(include='category').columns.tolist()

X_train_imp1 = pd.get_dummies(X_train_imp1, columns=cat_cols_imp1, drop_first=True)
X_test_imp1 = pd.get_dummies(X_test_imp1, columns=cat_cols_imp1, drop_first=True)

# Ensure columns match in train and test
X_test_imp1 = X_test_imp1.reindex(columns=X_train_imp1.columns, fill_value=0)

In [ ]:
y_test_imp1.value_counts()

In [ ]:
type(y_train_imp1)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_imp1 = LabelEncoder()
y_train_imp1 = le_imp1.fit_transform(y_train_imp1)
y_test_imp1 = le_imp1.transform(y_test_imp1)

print(le_imp1.classes_)  # To see which label is 0 and which is 1

y_train_imp1 = pd.Series(y_train_imp1)
y_test_imp1 = pd.Series(y_test_imp1)

In [ ]:
print(y_train_imp1.value_counts())
print(y_test_imp1.value_counts())

In [ ]:
!pip install catboost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import validation_curve, learning_curve, StratifiedKFold, cross_val_score
from sklearn.metrics import (f1_score, accuracy_score, precision_score, recall_score,
                           roc_auc_score, average_precision_score, confusion_matrix,
                           roc_curve, precision_recall_curve, classification_report)
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                            ExtraTreesClassifier, AdaBoostClassifier, BaggingClassifier,
                            VotingClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

# Example usage:


In [ ]:
%pip install xgboost

In [ ]:
%pip install lightgbm

In [ ]:
%pip install catboost

In [ ]:
def create_comprehensive_models_dict():
    """
    Create a comprehensive dictionary of all possible classification models
    """
    models_dict = {
        # Linear Models
        'Logistic Regression (L1)': LogisticRegression(
            C=1.0, penalty='l1', solver='liblinear', random_state=42, max_iter=1000
        ),
        'Logistic Regression (L2)': LogisticRegression(
            C=1.0, penalty='l2', solver='liblinear', random_state=42, max_iter=1000
        ),
        'Logistic Regression (ElasticNet)': LogisticRegression(
            C=1.0, penalty='elasticnet', l1_ratio=0.5, solver='saga', random_state=42, max_iter=1000
        ),
        'Ridge Classifier': RidgeClassifier(alpha=1.0, random_state=42),
        'SGD Classifier': SGDClassifier(loss='log_loss', alpha=0.0001, random_state=42, max_iter=1000),

        # Tree-based Models
        'Decision Tree': DecisionTreeClassifier(
            max_depth=10, min_samples_split=5, min_samples_leaf=2, random_state=42
        ),
        'Random Forest': RandomForestClassifier(
            n_estimators=100, max_depth=10, min_samples_split=5,
            min_samples_leaf=2, random_state=42
        ),
        'Extra Trees': ExtraTreesClassifier(
            n_estimators=100, max_depth=10, min_samples_split=5,
            min_samples_leaf=2, random_state=42
        ),
        'Gradient Boosting': GradientBoostingClassifier(
            n_estimators=100, learning_rate=0.1, max_depth=5,
            min_samples_split=5, min_samples_leaf=2, random_state=42
        ),
        'AdaBoost': AdaBoostClassifier(
            n_estimators=100, learning_rate=1.0, random_state=42
        ),
        'Bagging': BaggingClassifier(
            n_estimators=100, random_state=42
        ),

        # Advanced Gradient Boosting
        'XGBoost': XGBClassifier(
            n_estimators=100, max_depth=6, learning_rate=0.1,
            random_state=42, eval_metric='logloss'
        ),
        'LightGBM': LGBMClassifier(
            n_estimators=100, max_depth=6, learning_rate=0.1,
            random_state=42, verbose=-1
        ),
        'CatBoost': CatBoostClassifier(
            iterations=100, depth=6, learning_rate=0.1,
            random_state=42, verbose=False
        ),

        # Support Vector Machines
        'SVM (RBF)': SVC(
            C=1.0, kernel='rbf', probability=True, random_state=42
        ),
        'SVM (Linear)': SVC(
            C=1.0, kernel='linear', probability=True, random_state=42
        ),
        'SVM (Polynomial)': SVC(
            C=1.0, kernel='poly', degree=3, probability=True, random_state=42
        ),

        # Nearest Neighbors
        'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),

        # Naive Bayes
        'Gaussian Naive Bayes': GaussianNB(),
        'Multinomial Naive Bayes': MultinomialNB(),
        'Bernoulli Naive Bayes': BernoulliNB(),

        # Discriminant Analysis
        'Linear Discriminant Analysis': LinearDiscriminantAnalysis(),
        'Quadratic Discriminant Analysis': QuadraticDiscriminantAnalysis(),

        # Neural Networks
        'Multi-layer Perceptron': MLPClassifier(
            hidden_layer_sizes=(100, 50), max_iter=500, random_state=42
        ),

        # Baseline
        'Dummy Classifier (Stratified)': DummyClassifier(
            strategy='stratified', random_state=42
        ),
        'Dummy Classifier (Most Frequent)': DummyClassifier(
            strategy='most_frequent', random_state=42
        )
    }

    return models_dict

In [ ]:
def calculate_comprehensive_metrics(y_true, y_pred, y_pred_proba):
    """
    Calculate all classification metrics
    """
    return {
        'Test Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'F1 Score': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'ROC AUC': roc_auc_score(y_true, y_pred_proba) if len(np.unique(y_true)) == 2 else 0,
        'PR AUC': average_precision_score(y_true, y_pred_proba) if len(np.unique(y_true)) == 2 else 0
    }

In [ ]:
def plot_confusion_matrix(y_true, y_pred, model_name):
    """Plot confusion matrix"""
    plt.figure(figsize=(8, 6))
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()


In [ ]:
def plot_roc_curve(y_true, y_pred_proba, model_name):
    """Plot ROC curve"""
    if len(np.unique(y_true)) == 2:  # Binary classification only
        plt.figure(figsize=(8, 6))
        fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
        roc_auc = roc_auc_score(y_true, y_pred_proba)
        plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.3f})')
        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve - {model_name}')
        plt.legend()
        plt.show()

In [ ]:
def plot_precision_recall_curve(y_true, y_pred_proba, model_name):
    """Plot Precision-Recall curve"""
    if len(np.unique(y_true)) == 2:  # Binary classification only
        plt.figure(figsize=(8, 6))
        precision, recall, _ = precision_recall_curve(y_true, y_pred_proba)
        pr_auc = average_precision_score(y_true, y_pred_proba)
        plt.plot(recall, precision, label=f'PR curve (AUC = {pr_auc:.3f})')
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title(f'Precision-Recall Curve - {model_name}')
        plt.legend()
        plt.show()

In [ ]:
def plot_feature_importance(model, X_train, model_name):
    """Plot feature importance"""
    if hasattr(model, 'feature_importances_'):
        plt.figure(figsize=(10, 6))
        if hasattr(X_train, 'columns'):
            importances = pd.Series(model.feature_importances_, index=X_train.columns)
        else:
            importances = pd.Series(model.feature_importances_,
                                  index=[f'Feature_{i}' for i in range(len(model.feature_importances_))])
        importances.sort_values(ascending=False).head(10).plot(kind='bar')
        plt.title(f'Top 10 Feature Importances - {model_name}')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

In [ ]:
def detect_overfitting_comprehensive_enhanced(X_train, X_test, y_train, y_test, models_dict):
    """
    Enhanced comprehensive overfitting detection suite with improved analysis
    """
    results = []
    cv_results = {}
    overfitting_summary = {}

    # Stratified K-Fold Cross Validation
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for name, model in models_dict.items():
        print(f"\n{'='*60}")
        print(f"COMPREHENSIVE ANALYSIS: {name}")
        print(f"{'='*60}")

        try:
            # Fit model
            model.fit(X_train, y_train)

            # Predictions
            train_pred = model.predict(X_train)
            test_pred = model.predict(X_test)

            # Get prediction probabilities
            if hasattr(model, 'predict_proba'):
                train_pred_proba = model.predict_proba(X_train)[:, 1] if len(np.unique(y_test)) == 2 else model.predict_proba(X_train).max(axis=1)
                test_pred_proba = model.predict_proba(X_test)[:, 1] if len(np.unique(y_test)) == 2 else model.predict_proba(X_test).max(axis=1)
            else:
                train_pred_proba = model.decision_function(X_train) if hasattr(model, 'decision_function') else train_pred
                test_pred_proba = model.decision_function(X_test) if hasattr(model, 'decision_function') else test_pred

            # Calculate comprehensive metrics
            metrics = calculate_comprehensive_metrics(y_test, test_pred, test_pred_proba)

            # Enhanced overfitting detection
            overfitting_analysis = enhanced_overfitting_detection(
                y_train, y_test, train_pred, test_pred, train_pred_proba, test_pred_proba, name
            )

            # Print metrics
            print(f"PERFORMANCE METRICS:")
            for metric_name, value in metrics.items():
                print(f"   {metric_name}: {value:.4f}")

            print(f"\n ENHANCED OVERFITTING ANALYSIS:")
            print(f"   Train F1 Score: {overfitting_analysis['train_f1']:.4f}")
            print(f"   Test F1 Score: {overfitting_analysis['test_f1']:.4f}")
            print(f"   F1 Performance Gap: {overfitting_analysis['f1_gap']:.4f}")
            print(f"   Accuracy Gap: {overfitting_analysis['acc_gap']:.4f}")
            print(f"   Overfitting Score: {overfitting_analysis['overfitting_score']:.3f}")
            print(f"   Overfitting Risk: {overfitting_analysis['overfitting_risk']}")

            if overfitting_analysis['overfitting_reasons']:
                print(f"   Overfitting Indicators:")
                for reason in overfitting_analysis['overfitting_reasons']:
                    print(f"     • {reason}")

            # Store overfitting summary
            overfitting_summary[name] = {
                'risk': overfitting_analysis['overfitting_risk'],
                'score': overfitting_analysis['overfitting_score'],
                'reasons': overfitting_analysis['overfitting_reasons']
            }

            # Stratified K-Fold Cross Validation
            print(f"\nSTRATIFIED K-FOLD CROSS VALIDATION:")
            cv_scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1_weighted')
            cv_mean = cv_scores.mean()
            cv_std = cv_scores.std()

            print(f"   CV Scores: {[f'{score:.4f}' for score in cv_scores]}")
            print(f"   CV Mean: {cv_mean:.4f}")
            print(f"   CV Std: {cv_std:.4f}")
            print(f"   CV Coefficient of Variation: {cv_std/cv_mean:.4f}")

            if cv_std > 0.05:
                print("   High CV variance suggests instability/overfitting")

            cv_results[name] = cv_scores

            # Classification Report
            print(f"\n📋 DETAILED CLASSIFICATION REPORT:")
            print(classification_report(y_test, test_pred, zero_division=0))

            # Store results
            result_dict = {
                'Model': name,
                'Train F1': overfitting_analysis['train_f1'],
                'CV Mean F1': cv_mean,
                'CV Std F1': cv_std,
                'Overfitting Gap': overfitting_analysis['f1_gap'],
                'Overfitting Score': overfitting_analysis['overfitting_score'],
                'Overfitting Risk': overfitting_analysis['overfitting_risk']
            }
            result_dict.update(metrics)
            results.append(result_dict)

        except Exception as e:
            print(f"   Model {name} failed: {str(e)}")
            overfitting_summary[name] = {
                'risk': 'FAILED',
                'score': 1.0,
                'reasons': [f"Model failed to run: {str(e)}"]
            }
            continue

    return results, cv_results, overfitting_summary

In [ ]:
def healthcare_model_selection_algorithm(results):
    """
    Healthcare-specific model selection algorithm for kidney disease detection
    Priority: Recall > Precision > Stability > Low Overfitting > F1 > Accuracy
    """
    print(f"\n{'='*80}")
    print(f"HEALTHCARE MODEL SELECTION ALGORITHM")
    print(f"For Kidney Disease Detection - Minimizing False Negatives")
    print(f"{'='*80}")

    # Convert to DataFrame
    df = pd.DataFrame(results)

    # Healthcare-specific weights
    weights = {
        'Recall': 0.35,           # Highest priority - avoid missing kidney disease
        'Precision': 0.20,        # Important but secondary to recall
        'F1 Score': 0.15,         # Balanced metric
        'CV Mean F1': 0.15,       # Stability indicator
        'Test Accuracy': 0.10,    # Less important in healthcare
        'ROC AUC': 0.05,          # Additional metric
        'Overfitting_Penalty': -0.40,  # MUCH higher penalty for overfitting
        'Critical_Penalty': -0.60,     # Even higher penalty for critical overfitting
        'Stability_Bonus': 0.10        # Bonus for stable models
    }

    print("HEALTHCARE SELECTION CRITERIA:")
    print(" Recall (Sensitivity): 35% - Minimize false negatives")
    print(" Precision: 20% - Reduce false positives")
    print(" F1 Score: 15% - Balanced performance")
    print(" Cross-Validation Stability: 15% - Consistent performance")
    print(" Test Accuracy: 10% - Overall correctness")
    print("  ROC AUC: 5% - Additional validation")
    print("  Overfitting Penalty: -40% - Strong penalty for unreliable models")
    print("  Critical Overfitting Penalty: -60% - Severe penalty for suspicious models")
    print("  Stability Bonus: +10% - Reward for low variance")

    # Normalize metrics to 0-1 scale
    metrics_to_normalize = ['Test Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC', 'PR AUC', 'CV Mean F1']
    df_normalized = df.copy()

    for metric in metrics_to_normalize:
        if metric in df.columns:
            min_val = df[metric].min()
            max_val = df[metric].max()
            if max_val > min_val:
                df_normalized[metric] = (df[metric] - min_val) / (max_val - min_val)
            else:
                df_normalized[metric] = 1.0

    # Calculate penalties and bonuses
    df_normalized['Overfitting_Penalty'] = df['Overfitting Score'].apply(
        lambda x: -min(x, 1.0)  # Direct penalty based on overfitting score
    )

    df_normalized['Critical_Penalty'] = df['Overfitting Risk'].apply(
        lambda x: -1.0 if x == 'CRITICAL' else (-0.5 if x == 'HIGH' else 0)
    )

    df_normalized['Stability_Bonus'] = df['CV Std F1'].apply(
        lambda x: max(0, (0.05 - x) / 0.05) if x <= 0.05 else 0  # Bonus for low std
    )

    # Calculate weighted score
    df_normalized['Healthcare_Score'] = 0
    for metric, weight in weights.items():
        if metric in df_normalized.columns:
            df_normalized['Healthcare_Score'] += df_normalized[metric] * weight

    # Sort by healthcare score
    df_sorted = df_normalized.sort_values('Healthcare_Score', ascending=False)

    print(f"\nTOP 5 MODELS FOR HEALTHCARE APPLICATION:")
    print("-" * 100)
    for i, (_, row) in enumerate(df_sorted.head().iterrows(), 1):
        print(f"{i}. {row['Model']}")
        print(f"   Healthcare Score: {row['Healthcare_Score']:.4f}")
        print(f"   Recall: {row['Recall']:.4f} | Precision: {row['Precision']:.4f} | F1: {row['F1 Score']:.4f}")
        print(f"   Overfitting Risk: {row['Overfitting Risk']} | Score: {row['Overfitting Score']:.3f}")
        print(f"   CV Stability: {row['CV Mean F1']:.4f} ± {row['CV Std F1']:.4f}")
        print()

    best_model = df_sorted.iloc[0]['Model']
    best_score = df_sorted.iloc[0]['Healthcare_Score']

    print(f"RECOMMENDED MODEL FOR KIDNEY DISEASE DETECTION: {best_model}")
    print(f"   Healthcare Score: {best_score:.4f}")

    # Print models to avoid
    critical_models = df[df['Overfitting Risk'] == 'CRITICAL']['Model'].tolist()
    if critical_models:
        print(f"\nMODELS TO AVOID (Critical Overfitting Risk):")
        for model in critical_models:
            print(f"   {model}")

    return best_model, df_sorted

In [ ]:
def plot_model_comparison(results):
    """
    Plot comprehensive model comparison
    """
    # Create comparison DataFrame
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('F1 Score', ascending=False)

    # Plot model comparison
    plt.figure(figsize=(15, 8))
    metrics = ['Test Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC', 'PR AUC']
    x = np.arange(len(metrics))
    width = 0.15

    # Plot top 5 models
    top_models = results_df.head(5)
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

    for i, (_, row) in enumerate(top_models.iterrows()):
        values = [row[metric] for metric in metrics]
        plt.bar(x + i*width, values, width, label=row['Model'], color=colors[i])

    plt.xlabel('Metrics')
    plt.ylabel('Score')
    plt.title('Top 5 Models Comparison')
    plt.xticks(x + width*2, metrics, rotation=45, ha='right')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

    # Plot CV scores box plot
    plt.figure(figsize=(15, 8))
    cv_data = []
    model_names = []
    for _, row in results_df.head(10).iterrows():
        cv_data.append([row['CV Mean F1']] * 5)  # Simplified for visualization
        model_names.append(row['Model'])

    plt.boxplot(cv_data, labels=model_names)
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Cross-Validation F1 Score')
    plt.title('Cross-Validation Performance Comparison (Top 10 Models)')
    plt.tight_layout()
    plt.show()

In [ ]:
def validation_curve_analysis_enhanced(X_train, y_train, model, param_name, param_range):
    """
    Enhanced validation curve analysis that handles None values
    """
    # Handle None values in parameter range for plotting
    plot_param_range = []
    plot_labels = []

    for param in param_range:
        if param is None:
            plot_param_range.append(999)  # Use large number for None
            plot_labels.append('None')
        else:
            plot_param_range.append(param)
            plot_labels.append(str(param))

    try:
        train_scores, val_scores = validation_curve(
            model, X_train, y_train, param_name=param_name,
            param_range=param_range, cv=5, scoring='f1_weighted'
        )

        train_mean = np.mean(train_scores, axis=1)
        train_std = np.std(train_scores, axis=1)
        val_mean = np.mean(val_scores, axis=1)
        val_std = np.std(val_scores, axis=1)

        plt.figure(figsize=(10, 6))
        plt.plot(plot_param_range, train_mean, 'o-', label='Training Score')
        plt.plot(plot_param_range, val_mean, 'o-', label='Validation Score')
        plt.fill_between(plot_param_range, train_mean - train_std, train_mean + train_std, alpha=0.1)
        plt.fill_between(plot_param_range, val_mean - val_std, val_mean + val_std, alpha=0.1)

        # Set custom x-tick labels
        plt.xticks(plot_param_range, plot_labels)
        plt.xlabel(param_name)
        plt.ylabel('F1 Score')
        plt.title(f'Validation Curve - {param_name}')
        plt.legend()
        plt.grid(True)
        plt.show()
    except Exception as e:
        print(f"Validation curve analysis failed: {str(e)}")

In [ ]:
def data_leakage_detector(X_train, X_test, y_train, y_test):
    """
    Enhanced data leakage detection
    """
    print(f"\n{'='*50}")
    print("DATA LEAKAGE DETECTION")
    print(f"{'='*50}")

    # Check for identical samples
    if hasattr(X_train, 'values'):
        X_train_vals = X_train.values
        X_test_vals = X_test.values
    else:
        X_train_vals = X_train
        X_test_vals = X_test

    # Check for duplicate rows between train and test
    train_set = set([tuple(row) for row in X_train_vals])
    test_set = set([tuple(row) for row in X_test_vals])
    overlap = train_set.intersection(test_set)

    print(f"Identical samples between train/test: {len(overlap)}")
    if len(overlap) > 0:
        print("CRITICAL: Data leakage detected - identical samples in train/test!")
    else:
        print("No identical samples found between train/test sets")

    # Check feature correlation with target
    if hasattr(X_train, 'corrwith'):
        correlations = X_train.corrwith(pd.Series(y_train))
        high_corr = correlations[abs(correlations) > 0.9]
        if len(high_corr) > 0:
            print(f"Features with suspiciously high correlation (>0.9): {high_corr.to_dict()}")
        else:
            print("No suspiciously high feature-target correlations found")

    return len(overlap)

In [ ]:
def enhanced_overfitting_detection(y_train, y_test, train_pred, test_pred, train_pred_proba, test_pred_proba, model_name):
    """
    Enhanced overfitting detection with multiple indicators

    Returns:
    - overfitting_score: 0-1 score where 1 is maximum overfitting
    - overfitting_risk: LOW/MEDIUM/HIGH/CRITICAL
    - overfitting_reasons: List of specific reasons
    """
    reasons = []
    overfitting_indicators = []

    # 1. Performance Gap Analysis
    train_f1 = f1_score(y_train, train_pred, average='weighted', zero_division=0)
    test_f1 = f1_score(y_test, test_pred, average='weighted', zero_division=0)
    f1_gap = train_f1 - test_f1

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)
    acc_gap = train_acc - test_acc

    # 2. Perfect Score Detection (Highly Suspicious)
    perfect_train_score = (train_acc >= 0.999 or train_f1 >= 0.999)
    perfect_test_score = (test_acc >= 0.999 or test_f1 >= 0.999)

    # 3. Probability Distribution Analysis
    prob_variance_indicator = 0
    if hasattr(train_pred_proba, '__len__') and hasattr(test_pred_proba, '__len__'):
        try:
            # Check for extreme probability predictions
            train_extreme_probs = np.sum((train_pred_proba > 0.95) | (train_pred_proba < 0.05)) / len(train_pred_proba)
            test_extreme_probs = np.sum((test_pred_proba > 0.95) | (test_pred_proba < 0.05)) / len(test_pred_proba)

            # High confidence predictions that are wrong indicate overfitting
            if train_extreme_probs > 0.8:  # More than 80% extreme predictions
                prob_variance_indicator = 0.3
                reasons.append(f"High confidence predictions: {train_extreme_probs:.1%} of training predictions are >95% or <5% confident")
        except:
            pass

    # 4. Low Performance + High Gap = Learning Noise
    if test_f1 < 0.7 and f1_gap > 0.05:
        reasons.append(f"Low test performance ({test_f1:.3f}) with large gap ({f1_gap:.3f}) suggests learning noise")

    # Calculate overfitting score (0-1)
    gap_score = min(max(f1_gap, 0) / 0.2, 1.0)  # Normalize gap to 0-1 (0.2 = max reasonable gap)
    perfect_score_penalty = 0.8 if (perfect_train_score or perfect_test_score) else 0
    low_perf_high_gap_penalty = 0.6 if (test_f1 < 0.7 and f1_gap > 0.05) else 0

    overfitting_score = min(gap_score + perfect_score_penalty + low_perf_high_gap_penalty + prob_variance_indicator, 1.0)

    # Determine risk level
    if overfitting_score >= 0.8 or perfect_train_score or perfect_test_score:
        risk_level = "CRITICAL"
        if perfect_train_score:
            reasons.append("Perfect training score detected (99.9%+) - highly suspicious")
        if perfect_test_score:
            reasons.append("Perfect test score detected (99.9%+) - possible data leakage")
    elif overfitting_score >= 0.5:
        risk_level = "HIGH"
    elif overfitting_score >= 0.2:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    # Add gap-based reasons
    if f1_gap > 0.1:
        reasons.append(f"Very large F1 gap: {f1_gap:.3f} (train: {train_f1:.3f}, test: {test_f1:.3f})")
    elif f1_gap > 0.05:
        reasons.append(f"Large F1 gap: {f1_gap:.3f} (train: {train_f1:.3f}, test: {test_f1:.3f})")

    if acc_gap > 0.1:
        reasons.append(f"Very large accuracy gap: {acc_gap:.3f}")
    elif acc_gap > 0.05:
        reasons.append(f"Large accuracy gap: {acc_gap:.3f}")

    return {
        'overfitting_score': overfitting_score,
        'overfitting_risk': risk_level,
        'overfitting_reasons': reasons,
        'f1_gap': f1_gap,
        'acc_gap': acc_gap,
        'train_f1': train_f1,
        'test_f1': test_f1,
        'train_acc': train_acc,
        'test_acc': test_acc
    }


In [ ]:
def print_overfitting_summary(overfitting_summary):
    """
    Print a comprehensive overfitting risk summary
    """
    print(f"\n{'='*80}")
    print("OVERFITTING RISK SUMMARY")
    print(f"{'='*80}")

    # Group by risk level
    risk_groups = {'CRITICAL': [], 'HIGH': [], 'MEDIUM': [], 'LOW': [], 'FAILED': []}

    for model, info in overfitting_summary.items():
        risk_groups[info['risk']].append((model, info))

    # Print each risk category
    for risk_level in ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW', 'FAILED']:
        models = risk_groups[risk_level]
        if models:
            icon = {'CRITICAL': 'Critical', 'HIGH': 'High', 'MEDIUM': 'Medium', 'LOW': 'Low', 'FAILED': 'Failed'}[risk_level]
            print(f"\n{icon} {risk_level} OVERFITTING RISK ({len(models)} models):")
            print("-" * 60)

            for model, info in models:
                print(f"   • {model}")
                print(f"     Score: {info['score']:.3f}")
                if info['reasons']:
                    for reason in info['reasons'][:2]:  # Show first 2 reasons
                        print(f"     - {reason}")
                print()

    # Summary statistics
    total_models = len(overfitting_summary)
    safe_models = len(risk_groups['LOW'])
    risky_models = len(risk_groups['HIGH']) + len(risk_groups['CRITICAL'])

    print(f"  SUMMARY STATISTICS:")
    print(f"   Total Models Evaluated: {total_models}")
    print(f"   Safe Models (LOW risk): {safe_models} ({safe_models/total_models*100:.1f}%)")
    print(f"   Risky Models (HIGH/CRITICAL): {risky_models} ({risky_models/total_models*100:.1f}%)")
    print(f"   Failed Models: {len(risk_groups['FAILED'])}")

    return risk_groups


In [ ]:
def run_healthcare_ml_pipeline(X_train, X_test, y_train, y_test):
    """
    Complete healthcare ML pipeline with enhanced overfitting detection and comprehensive plotting
    """
    print(f"\n{'='*80}")
    print("HEALTHCARE ML ANALYSIS PIPELINE")
    print("Kidney Disease Detection - Enhanced Overfitting Detection")
    print(f"{'='*80}")

    # Convert target to binary if needed
    if hasattr(y_train, 'dtype') and y_train.dtype == 'object':
        unique_classes = np.unique(y_train)
        if len(unique_classes) == 2:
            y_train_processed = (y_train == unique_classes[0]).astype(int)
            y_test_processed = (y_test == unique_classes[0]).astype(int)
        else:
            from sklearn.preprocessing import LabelEncoder
            le = LabelEncoder()
            y_train_processed = le.fit_transform(y_train)
            y_test_processed = le.transform(y_test)
    else:
        y_train_processed = y_train
        y_test_processed = y_test

    print(f" Dataset Info:")
    print(f"   Training samples: {X_train.shape[0]}")
    print(f"   Test samples: {X_test.shape[0]}")
    print(f"   Features: {X_train.shape[1]}")
    print(f"   Classes: {len(np.unique(y_train_processed))}")
    print(f"   Class distribution: {dict(zip(*np.unique(y_train_processed, return_counts=True)))}")

    # Create models dictionary
    models = create_comprehensive_models_dict()
    print(f"\nTesting {len(models)} different classification algorithms...")

    # Run comprehensive analysis with enhanced overfitting detection
    results_list, cv_results, overfitting_summary = detect_overfitting_comprehensive_enhanced(
        X_train, X_test, y_train_processed, y_test_processed, models
    )

    # Convert results list to dictionary for easier access
    results_dict = {}
    for result in results_list:
        model_name = result['Model']
        results_dict[model_name] = result

    print(f"\n{'='*80}")
    print("INDIVIDUAL MODEL ANALYSIS WITH VISUALIZATIONS")
    print(f"{'='*80}")

    # Dictionary to store trained models for plotting
    trained_models = {}

    # Individual model analysis with plotting
    for model_name, model_results in results_dict.items():
        if model_results is None:
            continue

        print(f"\nANALYZING: {model_name}")
        print("-" * 60)

        try:
            # Get model instance
            if model_name not in models:
                print(f"Model {model_name} not found in models dictionary, skipping...")
                continue

            model = models[model_name]

            # Train the model
            model.fit(X_train, y_train_processed)
            trained_models[model_name] = model

            # Make predictions
            y_pred = model.predict(X_test)

            # Get prediction probabilities if available
            if hasattr(model, 'predict_proba'):
                y_pred_proba = model.predict_proba(X_test)
                if len(np.unique(y_test_processed)) == 2:
                    y_pred_proba = y_pred_proba[:, 1]
                else:
                    y_pred_proba = y_pred_proba.max(axis=1)
            elif hasattr(model, 'decision_function'):
                y_pred_proba = model.decision_function(X_test)
                # Normalize decision function scores to [0,1] for binary classification
                if len(np.unique(y_test_processed)) == 2:
                    y_pred_proba = (y_pred_proba - y_pred_proba.min()) / (y_pred_proba.max() - y_pred_proba.min())
            else:
                y_pred_proba = None

            print(f"Model Performance:")
            # Access metrics from model_results dictionary
            print(f"   Test Accuracy: {model_results.get('Test Accuracy', 0):.4f}")
            print(f"   Precision: {model_results.get('Precision', 0):.4f}")
            print(f"   Recall: {model_results.get('Recall', 0):.4f}")
            print(f"   F1 Score: {model_results.get('F1 Score', 0):.4f}")
            print(f"   ROC AUC: {model_results.get('ROC AUC', 0):.4f}")

            # Plot confusion matrix
            print(f"\n Generating Confusion Matrix...")
            plot_confusion_matrix(y_test_processed, y_pred, model_name)

            # Plot ROC curve (only for binary classification)
            if len(np.unique(y_test_processed)) == 2 and y_pred_proba is not None:
                print(f"Generating ROC Curve...")
                plot_roc_curve(y_test_processed, y_pred_proba, model_name)

                print(f"Generating Precision-Recall Curve...")
                plot_precision_recall_curve(y_test_processed, y_pred_proba, model_name)

            # Plot feature importance (if available)
            if hasattr(model, 'feature_importances_'):
                print(f"Generating Feature Importance Plot...")
                plot_feature_importance(model, X_train, model_name)
            elif hasattr(model, 'coef_') and model.coef_.ndim == 1:
                print(f"Generating Feature Coefficients Plot...")
                # Handle linear model coefficients
                plt.figure(figsize=(10, 6))
                if hasattr(X_train, 'columns'):
                    coef_series = pd.Series(np.abs(model.coef_), index=X_train.columns)
                else:
                    coef_series = pd.Series(np.abs(model.coef_), index=[f'Feature_{i}' for i in range(len(model.coef_))])
                coef_series.sort_values(ascending=False).head(10).plot(kind='bar')
                plt.title(f'Top 10 Feature Coefficients (Absolute) - {model_name}')
                plt.xticks(rotation=45, ha='right')
                plt.tight_layout()
                plt.show()

            # Validation curve analysis for selected models with hyperparameters
            print(f"Generating Validation Curve Analysis...")
            if model_name == 'Random Forest':
                validation_curve_analysis_enhanced(
                    X_train, y_train_processed, model,
                    'n_estimators', [10, 50, 100, 200, 300]
                )
            elif model_name == 'XGBoost':
                validation_curve_analysis_enhanced(
                    X_train, y_train_processed, model,
                    'max_depth', [3, 4, 5, 6, 7, 8]
                )
            elif model_name == 'LightGBM':
                validation_curve_analysis_enhanced(
                    X_train, y_train_processed, model,
                    'num_leaves', [10, 20, 30, 40, 50]
                )
            elif 'SVM' in model_name:
                validation_curve_analysis_enhanced(
                    X_train, y_train_processed, model,
                    'C', [0.1, 1, 10, 100, 1000]
                )
            elif 'Logistic Regression' in model_name:
                validation_curve_analysis_enhanced(
                    X_train, y_train_processed, model,
                    'C', [0.01, 0.1, 1, 10, 100]
                )

            print(f"Completed analysis for {model_name}\n")

        except Exception as e:
            print(f"Error analyzing {model_name}: {str(e)}")
            continue

    # Print overfitting summary
    risk_groups = print_overfitting_summary(overfitting_summary)

    # Convert results list to DataFrame for healthcare model selection
    results_df = pd.DataFrame(results_list)

    # Healthcare-specific model selection
    best_model_name, ranked_models = healthcare_model_selection_algorithm(results_df)

    # Generate comprehensive model comparison plots
    print(f"\n{'='*80}")
    print("COMPREHENSIVE MODEL COMPARISON VISUALIZATIONS")
    print(f"{'='*80}")

    print("Generating Top Models Comparison...")
    plot_model_comparison(results_df)

    # Final recommendations
    print(f"\n{'='*80}")
    print("FINAL HEALTHCARE RECOMMENDATIONS")
    print(f"{'='*80}")

    print(f"RECOMMENDED MODEL: {best_model_name}")
    best_stats = ranked_models.iloc[0]
    print(f"   Healthcare Score: {best_stats['Healthcare_Score']:.4f}")
    print(f"   Recall (Sensitivity): {best_stats['Recall']:.4f}")
    print(f"   Precision: {best_stats['Precision']:.4f}")
    print(f"   F1 Score: {best_stats['F1 Score']:.4f}")
    print(f"   Overfitting Risk: {best_stats['Overfitting Risk']}")

    print(f"\nTOP 3 SAFE MODELS FOR HEALTHCARE:")
    safe_models = ranked_models[ranked_models['Overfitting Risk'].isin(['LOW', 'MEDIUM'])].head(3)
    for i, (_, row) in enumerate(safe_models.iterrows(), 1):
        print(f"   {i}. {row['Model']} (Score: {row['Healthcare_Score']:.4f}, Risk: {row['Overfitting Risk']})")

    # Models to avoid
    avoid_models = ranked_models[ranked_models['Overfitting Risk'].isin(['CRITICAL', 'HIGH'])]['Model'].tolist()
    if avoid_models:
        print(f"\nMODELS TO AVOID IN HEALTHCARE:")
        for model in avoid_models[:5]:  # Show top 5 to avoid
            print(f"  {model}")

    # Additional comprehensive analysis plots
    print(f"\n GENERATING ADDITIONAL ANALYSIS PLOTS...")

    # Overfitting risk distribution plot
    plt.figure(figsize=(12, 8))
    risk_counts = ranked_models['Overfitting Risk'].value_counts()
    colors = {'LOW': 'green', 'MEDIUM': 'orange', 'HIGH': 'red', 'CRITICAL': 'darkred'}
    risk_colors = [colors.get(risk, 'gray') for risk in risk_counts.index]

    plt.subplot(2, 2, 1)
    plt.pie(risk_counts.values, labels=risk_counts.index, autopct='%1.1f%%', colors=risk_colors)
    plt.title('Overfitting Risk Distribution')

    # Healthcare scores distribution
    plt.subplot(2, 2, 2)
    plt.hist(ranked_models['Healthcare_Score'], bins=15, alpha=0.7, color='skyblue')
    plt.xlabel('Healthcare Score')
    plt.ylabel('Number of Models')
    plt.title('Healthcare Scores Distribution')

    # Recall vs Precision scatter plot
    plt.subplot(2, 2, 3)
    colors_risk = ranked_models['Overfitting Risk'].map(colors)
    plt.scatter(ranked_models['Recall'], ranked_models['Precision'], c=colors_risk, alpha=0.7)
    plt.xlabel('Recall (Sensitivity)')
    plt.ylabel('Precision')
    plt.title('Recall vs Precision (Colored by Risk)')

    # F1 Score vs CV Stability
    plt.subplot(2, 2, 4)
    plt.scatter(ranked_models['F1 Score'], ranked_models['CV Std F1'], c=colors_risk, alpha=0.7)
    plt.xlabel('F1 Score')
    plt.ylabel('CV Standard Deviation')
    plt.title('Performance vs Stability (Colored by Risk)')

    plt.tight_layout()
    plt.show()

    return {
        'results': results_dict,
        'results_list': results_list,
        'cv_results': cv_results,
        'overfitting_summary': overfitting_summary,
        'risk_groups': risk_groups,
        'best_model': best_model_name,
        'ranked_models': ranked_models,
        'safe_models': safe_models,
        'trained_models': trained_models
    }

In [ ]:
results = run_healthcare_ml_pipeline(X_train, X_test, y_train, y_test)


In [ ]:
print(results)

In [ ]:
results_imp1 = run_healthcare_ml_pipeline(X_train_imp1, X_test_imp1, y_train_imp1, y_test_imp1)


In [ ]:
X_train_imp1.isnull().sum()

In [ ]:
print(results_imp1)

In [ ]:
# Exclude target and categorical columns from features
feature_cols_imp2 = ['hemo','sg','sc','htn','bgr']

X_imp2 = df_clean[feature_cols_imp2]
y_imp2 = df_clean[target]

X_train_imp2, X_test_imp2, y_train_imp2, y_test_imp2 = train_test_split(X_imp2, y_imp2, test_size=0.2, random_state=42, stratify=y_imp2)

In [ ]:
y_test_imp2.value_counts()

In [ ]:
len(feature_cols_imp2)

In [ ]:
log_transform_cols_imp2 = ['bgr','sc']

for col in log_transform_cols_imp2:
    for df in [X_train_imp2, X_test_imp2]:
        df[col] = np.log(df[col])

In [ ]:
from sklearn.preprocessing import StandardScaler

# Identify numerical columns (excluding categorical and binned)
num_cols_imp2 = X_train_imp2.select_dtypes(include=['float64', 'int32']).columns.tolist()

scaler_imp2 = StandardScaler()
X_train_imp2[num_cols_imp2] = scaler_imp2.fit_transform(X_train_imp2[num_cols_imp2])
X_test_imp2[num_cols_imp2] = scaler_imp2.transform(X_test_imp2[num_cols_imp2])

In [ ]:
cat_cols_imp2 = X_train_imp2.select_dtypes(include='category').columns.tolist()

X_train_imp2 = pd.get_dummies(X_train_imp2, columns=cat_cols_imp2, drop_first=True)
X_test_imp2 = pd.get_dummies(X_test_imp2, columns=cat_cols_imp2, drop_first=True)

# Ensure columns match in train and test
X_test_imp2 = X_test_imp2.reindex(columns=X_train_imp2.columns, fill_value=0)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_imp2 = LabelEncoder()
y_train_imp2 = le_imp2.fit_transform(y_train_imp2)
y_test_imp2 = le_imp2.transform(y_test_imp2)

print(le_imp2.classes_)  # To see which label is 0 and which is 1

y_train_imp2 = pd.Series(y_train_imp2)
y_test_imp2 = pd.Series(y_test_imp2)

In [ ]:
X_test_imp2.isnull().sum().sum()

In [ ]:
results_imp2 = run_healthcare_ml_pipeline(X_train_imp2, X_test_imp2, y_train_imp2, y_test_imp2)


In [ ]:
print(results_imp2)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score, validation_curve
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier, BaggingClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

def get_hyperparameter_grids_for_small_dataset():
    """
    Hyperparameter grids specifically designed for datasets with ~400 records
    Focus on preventing overfitting through aggressive regularization
    """

    hyperparameter_grids = {

        # RANDOM FOREST - Healthcare optimized for minimal false negatives
        'Random Forest': {
            'model': RandomForestClassifier(random_state=42, class_weight='balanced'),
            'params': {
                'n_estimators': [10, 25, 50, 100],
                'max_depth': [3, 5, 7, 10, None],
                'min_samples_split': [20, 40, 60, 80],
                'min_samples_leaf': [10, 20, 30, 40],
                'max_features': ['sqrt', 'log2', 0.3, 0.5],
                'max_samples': [0.6, 0.7, 0.8, 0.9],
                'min_impurity_decrease': [0.0, 0.01, 0.02, 0.05],
                'ccp_alpha': [0.0, 0.01, 0.02, 0.05]
            }
        },

        # EXTRA TREES
        'Extra Trees': {
            'model': ExtraTreesClassifier(random_state=42, class_weight='balanced'),
            'params': {
                'n_estimators': [10, 25, 50],
                'max_depth': [3, 5, 7],
                'min_samples_split': [40, 60, 80, 100],
                'min_samples_leaf': [20, 30, 40, 50],
                'max_features': ['sqrt', 'log2', 0.3],
                'max_samples': [0.6, 0.7, 0.8],
                'min_impurity_decrease': [0.01, 0.02, 0.05],
                'ccp_alpha': [0.01, 0.02, 0.05, 0.1]
            }
        },

        # DECISION TREE
        'Decision Tree': {
            'model': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
            'params': {
                'max_depth': [3, 5, 7, 10],
                'min_samples_split': [40, 60, 80, 100],
                'min_samples_leaf': [20, 30, 40, 50],
                'max_features': ['sqrt', 'log2', 0.3, 0.5, None],
                'min_impurity_decrease': [0.01, 0.02, 0.05, 0.1],
                'ccp_alpha': [0.01, 0.02, 0.05, 0.1, 0.2]
            }
        },

        # GRADIENT BOOSTING
        'Gradient Boosting': {
            'model': GradientBoostingClassifier(random_state=42),
            'params': {
                'n_estimators': [10, 25, 50, 100],
                'learning_rate': [0.01, 0.05, 0.1, 0.2],
                'max_depth': [2, 3, 4, 5],
                'min_samples_split': [40, 60, 80],
                'min_samples_leaf': [20, 30, 40],
                'subsample': [0.6, 0.7, 0.8, 0.9],
                'max_features': ['sqrt', 'log2', 0.3, 0.5],
                'min_impurity_decrease': [0.01, 0.02, 0.05],
                'ccp_alpha': [0.0, 0.01, 0.02]
            }
        },

        # XGBOOST - Healthcare optimized
        'XGBoost': {
            'model': xgb.XGBClassifier(random_state=42, eval_metric='logloss'),
            'params': {
                'n_estimators': [50, 100,150,200],
                'learning_rate': [0.1, 0.15, 0.2, 0.3],
                'max_depth': [5,6,7,8],
                'min_child_weight': [ 3,5,7, 10],
                'subsample': [0.7, 0.8, 0.9, 1],
                'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
                'reg_alpha': [0, 0.01, 0.1, 1],
                'reg_lambda': [0.1, 1, 5, 10],
                'gamma': [0, 0.1, 0.5, 1],
                'scale_pos_weight': [1, 2, 3]  # Handle class imbalance
            }
        },

        # LIGHTGBM
        'LightGBM': {
            'model': lgb.LGBMClassifier(random_state=42, verbose=-1),
            'params': {
                'n_estimators': [10, 25, 50, 100],
                'learning_rate': [0.01, 0.05, 0.1, 0.2],
                'max_depth': [2, 3, 4, 5],
                'num_leaves': [7, 15, 31, 63],
                'min_child_samples': [10, 20, 30, 40],
                'min_split_gain': [0.01, 0.1, 0.5, 1],
                'subsample': [0.6, 0.7, 0.8, 0.9],
                'colsample_bytree': [0.3, 0.5, 0.7, 0.9],
                'reg_alpha': [0, 0.01, 0.1, 1],
                'reg_lambda': [0.1, 1, 5, 10],
                'min_child_weight': [0.001, 0.01, 0.1, 1],
                'class_weight': ['balanced', None]
            }
        },

        # CATBOOST
        'CatBoost': {
            'model': CatBoostClassifier(random_state=42, verbose=False),
            'params': {
                'iterations': [10, 25, 50, 100],
                'learning_rate': [0.01, 0.05, 0.1, 0.2],
                'depth': [2, 3, 4, 5],
                'min_data_in_leaf': [10, 20, 30, 40],
                'l2_leaf_reg': [1, 3, 5, 10, 20],
                'subsample': [0.6, 0.7, 0.8, 0.9],
                'colsample_bylevel': [0.3, 0.5, 0.7, 0.9],
                'border_count': [32, 64, 128],
                'bagging_temperature': [0, 0.5, 1],
                'class_weights': [[1, 1], [1, 2], [1, 3]]  # Handle imbalance
            }
        },

        # ADABOOST
        'AdaBoost': {
            'model': AdaBoostClassifier(random_state=42),
            'params': {
                'n_estimators': [10, 25, 50, 100],
                'learning_rate': [0.01, 0.1, 0.5, 1.0, 2.0],
                'algorithm': ['SAMME', 'SAMME.R']
            }
        },

        # BAGGING
        'Bagging': {
            'model': BaggingClassifier(random_state=42),
            'params': {
                'n_estimators': [10, 25, 50],
                'max_samples': [0.6, 0.7, 0.8, 0.9],
                'max_features': [0.3, 0.5, 0.7, 0.9],
                'bootstrap': [True, False],
                'bootstrap_features': [True, False]
            }
        },

        # LOGISTIC REGRESSION - Healthcare optimized
        'Logistic Regression': {
            'model': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
            'params': {
                'C': [0.001, 0.01, 0.1, 1, 10, 100],
                'penalty': ['l1', 'l2', 'elasticnet'],
                'solver': ['liblinear', 'saga'],
                'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],
                'fit_intercept': [True, False]
            }
        },

        # SGD CLASSIFIER
        'SGD Classifier': {
            'model': SGDClassifier(random_state=42, max_iter=1000, class_weight='balanced'),
            'params': {
                'alpha': [0.0001, 0.001, 0.01, 0.1, 1],
                'penalty': ['l1', 'l2', 'elasticnet'],
                'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],
                'learning_rate': ['constant', 'optimal', 'invscaling', 'adaptive'],
                'eta0': [0.001, 0.01, 0.1, 1],
                'early_stopping': [True, False],
                'validation_fraction': [0.1, 0.2, 0.3]
            }
        },

        # SVM RBF - Healthcare optimized
        'SVM (RBF)': {
            'model': SVC(random_state=42, probability=True, class_weight='balanced'),
            'params': {
                'C': [0.001, 0.01, 0.1, 1, 10, 100],
                'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
                'kernel': ['rbf'],
                'shrinking': [True, False],
                'cache_size': [200, 500, 1000]
            }
        },

        # SVM LINEAR
        'SVM (Linear)': {
            'model': SVC(random_state=42, probability=True, class_weight='balanced'),
            'params': {
                'C': [0.001, 0.01, 0.1, 1, 10, 100],
                'kernel': ['linear'],
                'shrinking': [True, False],
                'cache_size': [200, 500, 1000]
            }
        },

        # K-NEAREST NEIGHBORS
        'K-Nearest Neighbors': {
            'model': KNeighborsClassifier(),
            'params': {
                'n_neighbors': [3, 5, 7, 9, 11, 15, 21, 31],
                'weights': ['uniform', 'distance'],
                'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
                'metric': ['euclidean', 'manhattan', 'minkowski'],
                'p': [1, 2, 3],
                'leaf_size': [10, 20, 30, 40, 50]
            }
        },

        # LINEAR DISCRIMINANT ANALYSIS
        'Linear Discriminant Analysis': {
            'model': LinearDiscriminantAnalysis(),
            'params': {
                'solver': ['svd', 'lsqr', 'eigen'],
                'shrinkage': [None, 'auto', 0.1, 0.3, 0.5, 0.7, 0.9],
                'priors': [None],
                'n_components': [None, 1, 2, 3]
            }
        },

        # QUADRATIC DISCRIMINANT ANALYSIS
        'Quadratic Discriminant Analysis': {
            'model': QuadraticDiscriminantAnalysis(),
            'params': {
                'reg_param': [0.0, 0.01, 0.1, 0.3, 0.5, 0.7, 0.9],
                'store_covariance': [True, False],
                'tol': [1e-4, 1e-3, 1e-2]
            }
        },

        # MULTI-LAYER PERCEPTRON
        'Multi-layer Perceptron': {
            'model': MLPClassifier(random_state=42, max_iter=1000),
            'params': {
                'hidden_layer_sizes': [
                    (10,), (20,), (50,),
                    (10, 5), (20, 10), (50, 25),
                    (10, 10, 5), (20, 10, 5)
                ],
                'activation': ['tanh', 'relu', 'logistic'],
                'solver': ['adam', 'lbfgs'],
                'alpha': [0.0001, 0.001, 0.01, 0.1, 1],
                'learning_rate': ['constant', 'invscaling', 'adaptive'],
                'learning_rate_init': [0.001, 0.01, 0.1],
                'early_stopping': [True, False],
                'validation_fraction': [0.1, 0.2, 0.3],
                'beta_1': [0.9, 0.95, 0.99],
                'beta_2': [0.999, 0.9999]
            }
        }
    }

    return hyperparameter_grids

In [ ]:
def perform_healthcare_hyperparameter_tuning(X_train, y_train, model_name, scoring='recall', cv=5):
    """
    Perform healthcare-focused hyperparameter tuning prioritizing recall (minimizing false negatives)
    """
    grids = get_hyperparameter_grids_for_small_dataset()

    if model_name not in grids:
        print(f"Model {model_name} not found in hyperparameter grids")
        return None, None, None

    model_info = grids[model_name]
    model = model_info['model']
    param_grid = model_info['params']

    print(f"\nHEALTHCARE HYPERPARAMETER TUNING: {model_name}")
    print("-" * 70)
    print(f"   Primary Objective: Maximize Recall (Minimize False Negatives)")
    print(f"   Secondary Objective: Minimize Overfitting")
    print(f"   Cross-Validation Folds: {cv}")
    print(f"   Scoring Metric: {scoring}")

    param_combinations = np.prod([len(v) for v in param_grid.values()])
    print(f"   Parameter combinations to test: {param_combinations}")

    # Use RandomizedSearchCV for large parameter spaces to save computation time
    if param_combinations > 200:  # Increased threshold for better exploration
        n_iter = min(100, param_combinations)  # Test up to 100 combinations
        search = RandomizedSearchCV(
            model, param_grid, n_iter=n_iter, cv=cv, scoring=scoring,
            random_state=42, n_jobs=-1, verbose=0, return_train_score=True
        )
        search_type = f"RandomizedSearchCV (n_iter={n_iter})"
    else:
        search = GridSearchCV(
            model, param_grid, cv=cv, scoring=scoring,
            n_jobs=-1, verbose=0, return_train_score=True
        )
        search_type = "GridSearchCV (exhaustive)"

    print(f"   Search Strategy: {search_type}")

    # Fit the search
    print(f"   Training in progress...")
    search.fit(X_train, y_train)

    # Calculate overfitting metrics
    best_train_score = search.cv_results_['mean_train_score'][search.best_index_]
    best_val_score = search.best_score_
    overfitting_gap = best_train_score - best_val_score
    overfitting_ratio = overfitting_gap / best_train_score if best_train_score > 0 else 0

    print(f"\nHYPERPARAMETER TUNING RESULTS:")
    print(f"   Best {scoring.upper()} Score (CV): {best_val_score:.4f}")
    print(f"   Training Score: {best_train_score:.4f}")
    print(f"   Overfitting Gap: {overfitting_gap:.4f}")
    print(f"   Overfitting Ratio: {overfitting_ratio:.4f}")

    # Assess overfitting risk
    if overfitting_ratio > 0.15:
        overfitting_risk = "HIGH"
        risk_icon = "Risk"
    elif overfitting_ratio > 0.10:
        overfitting_risk = "MEDIUM"
        risk_icon = "Medium risk"
    else:
        overfitting_risk = "LOW"
        risk_icon = "Low risk"

    print(f"   {risk_icon} Overfitting Risk: {overfitting_risk}")

    print(f"\nBEST HYPERPARAMETERS:")
    for param, value in search.best_params_.items():
        print(f"  {param}: {value}")

    tuning_results = {
        'best_score': best_val_score,
        'train_score': best_train_score,
        'overfitting_gap': overfitting_gap,
        'overfitting_ratio': overfitting_ratio,
        'overfitting_risk': overfitting_risk,
        'search_type': search_type,
        'param_combinations_tested': n_iter if param_combinations > 200 else param_combinations
    }

    return search.best_estimator_, search.best_params_, tuning_results


In [ ]:
def create_tuned_models_dict(X_train, y_train):
    """
    Create a dictionary of hyperparameter-tuned models optimized for healthcare applications
    """
    print(f"\n{'='*80}")
    print("HEALTHCARE-OPTIMIZED HYPERPARAMETER TUNING")
    print("Optimizing for Minimal False Negatives in Kidney Disease Detection")
    print(f"{'='*80}")

    # Models to tune (prioritizing those most important for healthcare)
    priority_models = [
        'Random Forest',
        'XGBoost',
        'LightGBM',
        'Logistic Regression',
        'SVM (RBF)',
        'Gradient Boosting',
        'CatBoost'
    ]

    secondary_models = [
        'Decision Tree',
        'Extra Trees',
        'K-Nearest Neighbors',
        'Linear Discriminant Analysis',
        'Multi-layer Perceptron',
        'AdaBoost',
        'SGD Classifier'
    ]

    tuned_models = {}
    tuning_summary = {}

    # Tune priority models first
    print(f"\nTUNING PRIORITY MODELS (Healthcare Critical):")
    print("=" * 60)

    for model_name in priority_models:
        try:
            best_model, best_params, tuning_results = perform_healthcare_hyperparameter_tuning(
                X_train, y_train, model_name, scoring='recall'  # Prioritize recall for healthcare
            )

            if best_model is not None:
                tuned_models[model_name] = best_model
                tuning_summary[model_name] = {
                    'params': best_params,
                    'results': tuning_results
                }
                print(f"  {model_name}: Successfully tuned")
            else:
                print(f"   {model_name}: Tuning failed")

        except Exception as e:
            print(f"   {model_name}: Error during tuning - {str(e)}")
            continue

    # Tune secondary models
    print(f"\nTUNING SECONDARY MODELS:")
    print("=" * 60)

    for model_name in secondary_models:
        try:
            best_model, best_params, tuning_results = perform_healthcare_hyperparameter_tuning(
                X_train, y_train, model_name, scoring='recall'
            )

            if best_model is not None:
                tuned_models[model_name] = best_model
                tuning_summary[model_name] = {
                    'params': best_params,
                    'results': tuning_results
                }
                print(f"   {model_name}: Successfully tuned")
            else:
                print(f"   {model_name}: Tuning failed")

        except Exception as e:
            print(f"   {model_name}: Error during tuning - {str(e)}")
            continue

    # Add baseline models (no tuning needed)
    baseline_models = {
        'Gaussian Naive Bayes': GaussianNB(),
        'Multinomial Naive Bayes': MultinomialNB(),
        'Bernoulli Naive Bayes': BernoulliNB(),
        'Dummy Classifier (Stratified)': DummyClassifier(strategy='stratified', random_state=42),
        'Dummy Classifier (Most Frequent)': DummyClassifier(strategy='most_frequent', random_state=42)
    }

    tuned_models.update(baseline_models)

    print(f"\nHYPERPARAMETER TUNING SUMMARY:")
    print("=" * 60)
    print(f"   Priority Models Tuned: {len([m for m in priority_models if m in tuned_models])}/{len(priority_models)}")
    print(f"   Secondary Models Tuned: {len([m for m in secondary_models if m in tuned_models])}/{len(secondary_models)}")
    print(f"   Baseline Models Added: {len(baseline_models)}")
    print(f"   Total Models Ready: {len(tuned_models)}")

    return tuned_models, tuning_summary

In [ ]:
def plot_hyperparameter_tuning_summary(tuning_summary):
    """
    Create comprehensive visualization of hyperparameter tuning results
    """
    if not tuning_summary:
        print("No hyperparameter tuning results to plot.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    # Extract data
    model_names = list(tuning_summary.keys())
    scores = [info['results']['best_score'] for info in tuning_summary.values()]
    gaps = [info['results']['overfitting_gap'] for info in tuning_summary.values()]
    risks = [info['results']['overfitting_risk'] for info in tuning_summary.values()]

    # Color mapping for risks
    risk_colors = {'LOW': 'green', 'MEDIUM': 'orange', 'HIGH': 'red', 'CRITICAL': 'darkred'}
    colors = [risk_colors.get(risk, 'gray') for risk in risks]

    # 1. Recall Scores
    axes[0, 0].barh(range(len(model_names)), scores, color=colors, alpha=0.7)
    axes[0, 0].set_yticks(range(len(model_names)))
    axes[0, 0].set_yticklabels([name[:20] for name in model_names], fontsize=10)
    axes[0, 0].set_xlabel('Best CV Recall Score')
    axes[0, 0].set_title('Hyperparameter Tuning - Recall Scores')
    axes[0, 0].grid(True, alpha=0.3)

    # 2. Overfitting Gaps
    axes[0, 1].barh(range(len(model_names)), gaps, color=colors, alpha=0.7)
    axes[0, 1].set_yticks(range(len(model_names)))
    axes[0, 1].set_yticklabels([name[:20] for name in model_names], fontsize=10)
    axes[0, 1].set_xlabel('Overfitting Gap')
    axes[0, 1].set_title('Hyperparameter Tuning - Overfitting Gaps')
    axes[0, 1].grid(True, alpha=0.3)

    # 3. Risk Distribution
    risk_counts = pd.Series(risks).value_counts()
    axes[1, 0].pie(risk_counts.values, labels=risk_counts.index, autopct='%1.1f%%',
                   colors=[risk_colors.get(risk, 'gray') for risk in risk_counts.index])
    axes[1, 0].set_title('Overfitting Risk Distribution (Tuned Models)')

    # 4. Score vs Gap scatter
    scatter = axes[1, 1].scatter(scores, gaps, c=colors, alpha=0.7, s=100, edgecolors='black')
    axes[1, 1].set_xlabel('Best CV Recall Score')
    axes[1, 1].set_ylabel('Overfitting Gap')
    axes[1, 1].set_title('Performance vs Overfitting (Tuned Models)')
    axes[1, 1].grid(True, alpha=0.3)

    # Add annotations for best models
    for i, (score, gap, name) in enumerate(zip(scores, gaps, model_names)):
        if score > np.percentile(scores, 75) and gap < np.percentile(gaps, 50):
            axes[1, 1].annotate(name[:15], (score, gap), xytext=(5, 5),
                              textcoords='offset points', fontsize=8)

    plt.tight_layout()
    plt.show()

    # Print tuning statistics
    print(f"\nHYPERPARAMETER TUNING STATISTICS:")
    print(f"   Models Tuned: {len(tuning_summary)}")
    print(f"   Best Recall Score: {max(scores):.4f}")
    print(f"   Lowest Overfitting Gap: {min(gaps):.4f}")
    print(f"   Average Recall Improvement: {np.mean(scores):.4f}")
    print(f"   Average Overfitting Gap: {np.mean(gaps):.4f}")


In [ ]:
def print_hyperparameter_tuning_summary(tuning_summary):
    """
    Print a comprehensive summary of hyperparameter tuning results
    """
    print(f"\n{'='*80}")
    print("HYPERPARAMETER TUNING RESULTS SUMMARY")
    print(f"{'='*80}")

    if not tuning_summary:
        print("No hyperparameter tuning results available.")
        return

    # Sort by overfitting risk and best score
    risk_order = {'LOW': 1, 'MEDIUM': 2, 'HIGH': 3}
    sorted_models = sorted(tuning_summary.items(),
                          key=lambda x: (risk_order.get(x[1]['results']['overfitting_risk'], 4),
                                       -x[1]['results']['best_score']))

    print(f"\nMODELS RANKED BY HEALTHCARE SUITABILITY:")
    print("   (Ordered by: Low Overfitting Risk → High Recall Score)")
    print("-" * 80)

    for i, (model_name, info) in enumerate(sorted_models, 1):
        results = info['results']
        risk_icon = {'LOW': 'low', 'MEDIUM': 'medium', 'HIGH': 'high'}.get(results['overfitting_risk'], 'risk')

        print(f"\n{i}. {model_name}")
        print(f"   {risk_icon} Overfitting Risk: {results['overfitting_risk']}")
        print(f"   Best Recall Score: {results['best_score']:.4f}")
        print(f"   Train-Val Gap: {results['overfitting_gap']:.4f}")
        print(f"   Search Method: {results['search_type']}")
        print(f"   Combinations Tested: {results['param_combinations_tested']}")

        # Show key hyperparameters (limit to most important ones)
        key_params = list(info['params'].items())[:5]  # Show first 5 params
        if key_params:
            print(f"   Key Parameters:")
            for param, value in key_params:
                print(f"   {param}: {value}")

        if len(info['params']) > 5:
            print(f" ...{len(info['params']) - 5} more parameters")

    # Risk distribution summary
    risk_counts = {}
    for model_name, info in tuning_summary.items():
        risk = info['results']['overfitting_risk']
        risk_counts[risk] = risk_counts.get(risk, 0) + 1

    print(f"\nOVERFITTING RISK DISTRIBUTION:")
    for risk, count in sorted(risk_counts.items(), key=lambda x: risk_order.get(x[0], 4)):
        icon = {'LOW': 'low', 'MEDIUM': 'medium', 'HIGH': 'high'}.get(risk, 'risk')
        percentage = count / len(tuning_summary) * 100
        print(f"   {icon} {risk}: {count} models ({percentage:.1f}%)")

    # Best performing models by metric
    best_recall = max(tuning_summary.items(), key=lambda x: x[1]['results']['best_score'])
    lowest_overfitting = min(tuning_summary.items(), key=lambda x: x[1]['results']['overfitting_gap'])

    print(f"\nSTANDOUT MODELS:")
    print(f"   Highest Recall: {best_recall[0]} ({best_recall[1]['results']['best_score']:.4f})")
    print(f"   Lowest Overfitting: {lowest_overfitting[0]} (Gap: {lowest_overfitting[1]['results']['overfitting_gap']:.4f})")

In [ ]:
def run_healthcare_ml_pipeline_hyp(X_train, X_test, y_train, y_test):
    """
    Complete healthcare ML pipeline with enhanced hyperparameter tuning and overfitting detection
    """
    print(f"\n{'='*80}")
    print("HEALTHCARE ML ANALYSIS PIPELINE")
    print("Kidney Disease Detection - Enhanced with Hyperparameter Tuning")
    print(f"{'='*80}")

    # Convert target to binary if needed
    if hasattr(y_train, 'dtype') and y_train.dtype == 'object':
        unique_classes = np.unique(y_train)
        if len(unique_classes) == 2:
            y_train_processed = (y_train == unique_classes[0]).astype(int)
            y_test_processed = (y_test == unique_classes[0]).astype(int)
        else:
            from sklearn.preprocessing import LabelEncoder
            le = LabelEncoder()
            y_train_processed = le.fit_transform(y_train)
            y_test_processed = le.transform(y_test)
    else:
        y_train_processed = y_train
        y_test_processed = y_test

    print(f"   Dataset Info:")
    print(f"   Training samples: {X_train.shape[0]}")
    print(f"   Test samples: {X_test.shape[0]}")
    print(f"   Features: {X_train.shape[1]}")
    print(f"   Classes: {len(np.unique(y_train_processed))}")
    print(f"   Class distribution: {dict(zip(*np.unique(y_train_processed, return_counts=True)))}")

    # Create hyperparameter-tuned models
    models, tuning_summary = create_tuned_models_dict(X_train, y_train_processed)
    print(f"\nTesting {len(models)} different classification algorithms...")
    print(f"   {len(tuning_summary)} models have been hyperparameter-tuned")
    print(f"   {len(models) - len(tuning_summary)} baseline models included")

    # Print hyperparameter tuning summary
    if tuning_summary:
        print_hyperparameter_tuning_summary(tuning_summary)

    # Run comprehensive analysis with enhanced overfitting detection
    results_list, cv_results, overfitting_summary = detect_overfitting_comprehensive_enhanced(
        X_train, X_test, y_train_processed, y_test_processed, models
    )

    # Convert results list to dictionary for easier access
    results_dict = {}
    for result in results_list:
        model_name = result['Model']
        results_dict[model_name] = result

    print(f"\n{'='*80}")
    print("INDIVIDUAL MODEL ANALYSIS WITH VISUALIZATIONS")
    print(f"{'='*80}")

    # Dictionary to store trained models for plotting
    trained_models = {}

    # Individual model analysis with plotting
    for model_name, model_results in results_dict.items():
        if model_results is None:
            continue
        print(f"\n ANALYZING: {model_name}")
        print("-" * 60)

        try:
            # Get model instance
            if model_name not in models:
                print(f"Model {model_name} not found in models dictionary, skipping...")
                continue

            model = models[model_name]

            # Train the model
            model.fit(X_train, y_train_processed)
            trained_models[model_name] = model

            # Make predictions
            y_pred = model.predict(X_test)

            # Get prediction probabilities if available
            if hasattr(model, 'predict_proba'):
                y_pred_proba = model.predict_proba(X_test)
                if len(np.unique(y_test_processed)) == 2:
                    y_pred_proba = y_pred_proba[:, 1]
                else:
                    y_pred_proba = y_pred_proba.max(axis=1)
            elif hasattr(model, 'decision_function'):
                y_pred_proba = model.decision_function(X_test)
                # Normalize decision function scores to [0,1] for binary classification
                if len(np.unique(y_test_processed)) == 2:
                    y_pred_proba = (y_pred_proba - y_pred_proba.min()) / (y_pred_proba.max() - y_pred_proba.min())
            else:
                y_pred_proba = None

            print(f"Model Performance:")
            # Access metrics from model_results dictionary
            print(f"   Test Accuracy: {model_results.get('Test Accuracy', 0):.4f}")
            print(f"   Precision: {model_results.get('Precision', 0):.4f}")
            print(f"   Recall: {model_results.get('Recall', 0):.4f}")
            print(f"   F1 Score: {model_results.get('F1 Score', 0):.4f}")
            print(f"   ROC AUC: {model_results.get('ROC AUC', 0):.4f}")

            # Display hyperparameter tuning results if available
            if model_name in tuning_summary:
                tuning_info = tuning_summary[model_name]
                print(f"\nHyperparameter Tuning Results:")
                print(f"  Best Recall Score (CV): {tuning_info['results']['best_score']:.4f}")
                print(f" Overfitting Risk: {tuning_info['results']['overfitting_risk']}")
                print(f" Overfitting Gap: {tuning_info['results']['overfitting_gap']:.4f}")
                print(f" Search Method: {tuning_info['results']['search_type']}")

                # Show key hyperparameters
                key_params = list(tuning_info['params'].items())[:3]  # Show first 3 params
                if key_params:
                    print(f"Key Tuned Parameters:")
                    for param, value in key_params:
                        print(f"{param}: {value}")

            # Plot confusion matrix
            print(f"\nGenerating Confusion Matrix...")
            plot_confusion_matrix(y_test_processed, y_pred, model_name)

            # Plot ROC curve (only for binary classification)
            if len(np.unique(y_test_processed)) == 2 and y_pred_proba is not None:
                print(f"Generating ROC Curve...")
                plot_roc_curve(y_test_processed, y_pred_proba, model_name)

                print(f"Generating Precision-Recall Curve...")
                plot_precision_recall_curve(y_test_processed, y_pred_proba, model_name)

            # Plot feature importance (if available)
            if hasattr(model, 'feature_importances_'):
                print(f"Generating Feature Importance Plot...")
                plot_feature_importance(model, X_train, model_name)
            elif hasattr(model, 'coef_') and model.coef_.ndim == 1:
                print(f"Generating Feature Coefficients Plot...")
                # Handle linear model coefficients
                plt.figure(figsize=(10, 6))
                if hasattr(X_train, 'columns'):
                    coef_series = pd.Series(np.abs(model.coef_), index=X_train.columns)
                else:
                    coef_series = pd.Series(np.abs(model.coef_), index=[f'Feature_{i}' for i in range(len(model.coef_))])
                coef_series.sort_values(ascending=False).head(10).plot(kind='bar')
                plt.title(f'Top 10 Feature Coefficients (Absolute) - {model_name}')
                plt.xticks(rotation=45, ha='right')
                plt.tight_layout()
                plt.show()

            # Validation curve analysis for selected models with hyperparameters
            print(f"Generating Validation Curve Analysis...")
            if model_name == 'Random Forest':
                validation_curve_analysis_enhanced(
                    X_train, y_train_processed, model,
                    'n_estimators', [10, 50, 100, 200, 300]
                )
            elif model_name == 'XGBoost':
                validation_curve_analysis_enhanced(
                    X_train, y_train_processed, model,
                    'max_depth', [3, 4, 5, 6, 7, 8]
                )
            elif model_name == 'LightGBM':
                validation_curve_analysis_enhanced(
                    X_train, y_train_processed, model,
                    'num_leaves', [10, 20, 30, 40, 50]
                )
            elif 'SVM' in model_name:
                validation_curve_analysis_enhanced(
                    X_train, y_train_processed, model,
                    'C', [0.1, 1, 10, 100, 1000]
                )
            elif 'Logistic Regression' in model_name:
                validation_curve_analysis_enhanced(
                    X_train, y_train_processed, model,
                    'C', [0.01, 0.1, 1, 10, 100]
                )

            print(f"Completed analysis for {model_name}\n")

        except Exception as e:
            print(f"Error analyzing {model_name}: {str(e)}")
            continue

        # Print overfitting summary
    risk_groups = print_overfitting_summary(overfitting_summary)

    # Convert results list to DataFrame for healthcare model selection
    results_df = pd.DataFrame(results_list)

    # Healthcare-specific model selection
    best_model_name, ranked_models = healthcare_model_selection_algorithm(results_df)

    # Generate comprehensive model comparison plots
    print(f"\n{'='*80}")
    print("COMPREHENSIVE MODEL COMPARISON VISUALIZATIONS")
    print(f"{'='*80}")

    print("Generating Top Models Comparison...")
    plot_model_comparison(results_df)

    # Enhanced hyperparameter tuning summary visualization
    if tuning_summary:
        print(f"\nGenerating Hyperparameter Tuning Summary Visualization...")
        plot_hyperparameter_tuning_summary(tuning_summary)

    # Final recommendations with hyperparameter considerations
    print(f"\n{'='*80}")
    print(" FINAL HEALTHCARE RECOMMENDATIONS")
    print(f"{'='*80}")

    print(f"RECOMMENDED MODEL: {best_model_name}")
    best_stats = ranked_models.iloc[0]
    print(f"   Healthcare Score: {best_stats['Healthcare_Score']:.4f}")
    print(f"   Recall (Sensitivity): {best_stats['Recall']:.4f}")
    print(f"   Precision: {best_stats['Precision']:.4f}")
    print(f"   F1 Score: {best_stats['F1 Score']:.4f}")
    print(f"   Overfitting Risk: {best_stats['Overfitting Risk']}")

    # Show hyperparameter tuning info for best model if available
    if best_model_name in tuning_summary:
        best_tuning = tuning_summary[best_model_name]
        print(f"   Hyperparameter Optimization:")
        print(f"   Tuning Method: {best_tuning['results']['search_type']}")
        print(f"   CV Recall Score: {best_tuning['results']['best_score']:.4f}")
        print(f"   Overfitting Gap: {best_tuning['results']['overfitting_gap']:.4f}")

    print(f"\ TOPn 3 SAFE MODELS FOR HEALTHCARE:")
    safe_models = ranked_models[ranked_models['Overfitting Risk'].isin(['LOW', 'MEDIUM'])].head(3)
    for i, (_, row) in enumerate(safe_models.iterrows(), 1):
        risk_indicator = "Low" if row['Overfitting Risk'] == 'LOW' else "high"
        tuning_indicator = " " if row['Model'] in tuning_summary else " "
        print(f"   {i}. {tuning_indicator} {row['Model']} (Score: {row['Healthcare_Score']:.4f}, Risk: {risk_indicator}{row['Overfitting Risk']})")

    # Models to avoid with hyperparameter tuning context
    avoid_models = ranked_models[ranked_models['Overfitting Risk'].isin(['CRITICAL', 'HIGH'])]['Model'].tolist()
    if avoid_models:
        print(f"\nMODELS TO AVOID IN HEALTHCARE:")
        for model in avoid_models[:5]:  # Show top 5 to avoid
            if model in tuning_summary:
                gap = tuning_summary[model]['results']['overfitting_gap']
                print(f"    {model} (Overfitting Gap: {gap:.4f})")
            else:
                print(f"   {model}")

    # Hyperparameter tuning insights
    if tuning_summary:
        print(f"\nHYPERPARAMETER TUNING INSIGHTS:")

        # Count tuned models by risk level
        tuned_risks = {}
        for model_name, info in tuning_summary.items():
            risk = info['results']['overfitting_risk']
            tuned_risks[risk] = tuned_risks.get(risk, 0) + 1

        print(f"  Tuned Models by Risk Level:")
        risk_order = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
        for risk in risk_order:
            if risk in tuned_risks:
                icon = {'LOW': 'l', 'MEDIUM': 'm', 'HIGH': 'h', 'CRITICAL': 'c'}.get(risk, 'r')
                print(f"      {icon} {risk}: {tuned_risks[risk]} models")

        # Best tuning results
        best_tuned_recall = max(tuning_summary.items(), key=lambda x: x[1]['results']['best_score'])
        lowest_overfitting_tuned = min(tuning_summary.items(), key=lambda x: x[1]['results']['overfitting_gap'])

        print(f"   Best Tuned Recall: {best_tuned_recall[0]} ({best_tuned_recall[1]['results']['best_score']:.4f})")
        print(f"    Lowest Overfitting (Tuned): {lowest_overfitting_tuned[0]} (Gap: {lowest_overfitting_tuned[1]['results']['overfitting_gap']:.4f})")

    # Additional comprehensive analysis plots
    print(f"\nGENERATING ADDITIONAL ANALYSIS PLOTS...")

    # Overfitting risk distribution plot
    plt.figure(figsize=(15, 10))

    # Risk distribution
    plt.subplot(2, 3, 1)
    risk_counts = ranked_models['Overfitting Risk'].value_counts()
    colors = {'LOW': 'green', 'MEDIUM': 'orange', 'HIGH': 'red', 'CRITICAL': 'darkred'}
    risk_colors = [colors.get(risk, 'gray') for risk in risk_counts.index]
    plt.pie(risk_counts.values, labels=risk_counts.index, autopct='%1.1f%%', colors=risk_colors)
    plt.title('Overfitting Risk Distribution')

    # Healthcare scores distribution
    plt.subplot(2, 3, 2)
    plt.hist(ranked_models['Healthcare_Score'], bins=15, alpha=0.7, color='skyblue', edgecolor='black')
    plt.xlabel('Healthcare Score')
    plt.ylabel('Number of Models')
    plt.title('Healthcare Scores Distribution')
    plt.grid(True, alpha=0.3)

    # Recall vs Precision scatter plot
    plt.subplot(2, 3, 3)
    colors_risk = ranked_models['Overfitting Risk'].map(colors)
    scatter = plt.scatter(ranked_models['Recall'], ranked_models['Precision'],
                         c=colors_risk, alpha=0.7, s=60, edgecolors='black', linewidth=0.5)
    plt.xlabel('Recall (Sensitivity)')
    plt.ylabel('Precision')
    plt.title('Recall vs Precision (Colored by Risk)')
    plt.grid(True, alpha=0.3)

    # F1 Score vs CV Stability
    plt.subplot(2, 3, 4)
    plt.scatter(ranked_models['F1 Score'], ranked_models['CV Std F1'],
               c=colors_risk, alpha=0.7, s=60, edgecolors='black', linewidth=0.5)
    plt.xlabel('F1 Score')
    plt.ylabel('CV Standard Deviation')
    plt.title('Performance vs Stability (Colored by Risk)')
    plt.grid(True, alpha=0.3)

    # Hyperparameter tuning comparison (if available)
    if tuning_summary:
        plt.subplot(2, 3, 5)
        tuned_models_data = []
        tuned_scores = []
        tuned_gaps = []
        for model_name in ranked_models['Model']:
            if model_name in tuning_summary:
                tuned_models_data.append(model_name[:15])  # Truncate long names
                tuned_scores.append(tuning_summary[model_name]['results']['best_score'])
                tuned_gaps.append(tuning_summary[model_name]['results']['overfitting_gap'])

        if tuned_models_data:
            plt.scatter(tuned_scores, tuned_gaps, alpha=0.7, s=60,
                       c='purple', edgecolors='black', linewidth=0.5)
            plt.xlabel('Tuned CV Recall Score')
            plt.ylabel('Overfitting Gap')
            plt.title('Hyperparameter Tuning Results')
            plt.grid(True, alpha=0.3)

            # Add model names as annotations for top performers
            for i, (score, gap, name) in enumerate(zip(tuned_scores, tuned_gaps, tuned_models_data)):
                if score > np.percentile(tuned_scores, 75) and gap < np.percentile(tuned_gaps, 50):
                    plt.annotate(name, (score, gap), xytext=(5, 5),
                               textcoords='offset points', fontsize=8)

    # Model complexity vs performance
    plt.subplot(2, 3, 6)
    # Create a complexity score based on model type
    complexity_map = {
        'Dummy': 1, 'Naive Bayes': 2, 'Logistic Regression': 3, 'LDA': 3, 'QDA': 4,
        'Decision Tree': 4, 'KNN': 4, 'SVM': 5, 'Random Forest': 6, 'Extra Trees': 6,
        'AdaBoost': 6, 'Gradient Boosting': 7, 'XGBoost': 8, 'LightGBM': 8, 'CatBoost': 8,
        'MLP': 9, 'SGD': 3, 'Ridge': 3, 'Bagging': 5
    }

    complexity_scores = []
    for model_name in ranked_models['Model']:
        complexity = 5  # default
        for key, value in complexity_map.items():
            if key.lower() in model_name.lower():
                complexity = value
                break
        complexity_scores.append(complexity)

    plt.scatter(complexity_scores, ranked_models['Healthcare_Score'],
               c=colors_risk, alpha=0.7, s=60, edgecolors='black', linewidth=0.5)
    plt.xlabel('Model Complexity')
    plt.ylabel('Healthcare Score')
    plt.title('Complexity vs Healthcare Performance')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Summary statistics
    print(f"\n SUMMARY STATISTICS:")
    print(f"   Total Models Evaluated: {len(results_list)}")
    print(f"   Models with Hyperparameter Tuning: {len(tuning_summary)}")
    print(f"   Low Risk Models: {len(ranked_models[ranked_models['Overfitting Risk'] == 'LOW'])}")
    print(f"   Medium Risk Models: {len(ranked_models[ranked_models['Overfitting Risk'] == 'MEDIUM'])}")
    print(f"   High Risk Models: {len(ranked_models[ranked_models['Overfitting Risk'] == 'HIGH'])}")
    print(f"   Critical Risk Models: {len(ranked_models[ranked_models['Overfitting Risk'] == 'CRITICAL'])}")
    print(f"   Average Healthcare Score: {ranked_models['Healthcare_Score'].mean():.4f}")
    print(f"   Average Recall: {ranked_models['Recall'].mean():.4f}")
    print(f"   Average Precision: {ranked_models['Precision'].mean():.4f}")

    return {
        'results': results_dict,
        'results_list': results_list,
        'cv_results': cv_results,
        'overfitting_summary': overfitting_summary,
        'risk_groups': risk_groups,
        'best_model': best_model_name,
        'ranked_models': ranked_models,
        'safe_models': safe_models,
        'trained_models': trained_models,
        'tuning_summary': tuning_summary,
        'hyperparameter_insights': {
            'tuned_models_count': len(tuning_summary),
            'best_tuned_recall': best_tuned_recall if tuning_summary else None,
            'lowest_overfitting_tuned': lowest_overfitting_tuned if tuning_summary else None,
            'risk_distribution': tuned_risks if tuning_summary else None
        }
    }



In [ ]:
run_healthcare_ml_pipeline_hyp(X_train,X_test,y_train,y_test)

In [ ]:
run_healthcare_ml_pipeline_hyp(X_train_imp1, X_test_imp1, y_train_imp1, y_test_imp1)

In [ ]:
run_healthcare_ml_pipeline_hyp(X_train_imp2, X_test_imp2, y_train_imp2, y_test_imp2)

In [ ]:
print(df_clean.info())

In [ ]:
df_out=pd.read_csv('ckd_synthetic_data.csv')

In [ ]:
df_out.isnull().sum()

In [ ]:
df_out.info()

In [ ]:
df_out.describe()

In [ ]:
# Avoid division by zero or negative values
df_out['eGFR'] = 186 * (df_out['sc'].clip(lower=0.01))**(-1.154) * (df_out['age'].clip(lower=1))**(-0.203)

In [ ]:
df_out['comorb_score'] = (
    (df_out['htn'] == 'yes').astype(int) +
    (df_out['dm'] == 'yes').astype(int) +
    (df_out['cad'] == 'yes').astype(int)
)

In [ ]:
# Calculate z-scores (lower = more severe anemia)
df_out['anemia_severity'] = -(
    zscore(df_out['hemo']) +
    zscore(df_out['pcv']) +
    zscore(df_out['rc'])
)

In [ ]:
df_out['kidney_func_score'] = (
    zscore(df_out['bu']) +
    zscore(df_out['sc']) -
    zscore(df_out['sod'])
)

In [ ]:
df_out['symptom_severity'] = (
    (df_out['appet'] == 'poor').astype(int) +
    (df_out['pe'] == 'yes').astype(int) +
    (df_out['ane'] == 'yes').astype(int)
)

In [ ]:
df_out.value_counts()

In [ ]:
c=0
for col in df_out.select_dtypes(include='object').columns:
    df_out[col] = df_out[col].astype('category')
    c+=1
print(c)

In [ ]:
df_out.drop(columns=['id'],inplace=True)

In [ ]:
print('New df:')
print(len(df_out.columns))
print()
print('Old df:')
print(len(df_clean.columns))

In [ ]:
df_out['al'] = pd.Categorical(df_out['al'], categories=[0,1,2,3,4,5], ordered=True)
df_out['su'] = pd.Categorical(df_out['su'], categories=[0,1,2,3,4,5], ordered=True)

In [ ]:
# Exclude target and categorical columns from features

X_out = df_out[feature_cols]
y_out = df_out[target]

In [ ]:
X_out.columns.tolist()

In [ ]:
print(X_out.head())

In [ ]:
log_transform_cols = ['bgr', 'bu', 'sc', 'wc', 'eGFR']

for col in log_transform_cols:
  X_out[col] = np.log(X_out[col])

In [ ]:
print(X_out.head())

In [ ]:
# Identify numerical columns (excluding categorical and binned)

X_out[num_cols] = scaler.transform(X_out[num_cols])

In [ ]:
print(X_out.head())

In [ ]:
X_out = pd.get_dummies(X_out, columns=cat_cols, drop_first=True)

# Ensure columns match in train and test
X_out = X_out.reindex(columns=X_train.columns, fill_value=0)

In [ ]:
print(X_out.head())

In [ ]:
le

In [ ]:
y_out = le.transform(y_out)

y_out = pd.Series(y_out)

In [ ]:
print(X_out.head())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
import xgboost as xgb
import lightgbm as lgb
import pickle
import warnings
warnings.filterwarnings('ignore')

def train_and_evaluate_models(X_train, y_train, test_X, test_y):
    """
    Trains multiple models with specified hyperparameters and evaluates them on test data.

    Parameters:
    -----------
    X_train : array-like or DataFrame
        Training features
    y_train : array-like or Series
        Training target variable
    test_X : array-like or DataFrame
        Test features for evaluation
    test_y : array-like or Series
        Test target variable for evaluation

    Returns:
    --------
    tuple : (leaderboard, trained_models, saved_models)
        - leaderboard: pandas.DataFrame with model rankings and accuracies
        - trained_models: dict with all trained model objects
        - saved_models: dict with model names and their saved filenames
    """

    # Define model hyperparameters dictionary
    model_configs = {
        'Logistic Regression': {
            'model': LogisticRegression,
            'params': {
                'solver': 'liblinear',
                'penalty': 'l2',
                'C': 0.001,
                'fit_intercept': False,
                'random_state': 42
            }
        },
        'XGBoost': {
            'model': xgb.XGBClassifier,
            'params': {
                'subsample': 0.9,
                'scale_pos_weight': 1,
                'reg_lambda': 10,
                'reg_alpha': 0,
                'n_estimators': 200,
                'min_child_weight': 7,
                'max_depth': 8,
                'learning_rate': 0.15,
                'gamma': 0.5,
                'colsample_bytree': 0.6,
                'random_state': 42,
                'eval_metric': 'logloss'  # To suppress warnings
            }
        },
        'Gradient Boosting': {
            'model': GradientBoostingClassifier,
            'params': {
                'subsample': 0.6,
                'n_estimators': 100,
                'min_samples_split': 80,
                'min_samples_leaf': 40,
                'min_impurity_decrease': 0.05,
                'max_features': 'log2',
                'max_depth': 2,
                'learning_rate': 0.01,
                'ccp_alpha': 0.0,
                'random_state': 42
            }
        },
        'Light GBM': {
            'model': lgb.LGBMClassifier,
            'params': {
                'subsample': 0.9,
                'reg_lambda': 0.1,
                'reg_alpha': 1,
                'num_leaves': 31,
                'n_estimators': 25,
                'min_split_gain': 1,
                'min_child_weight': 1,
                'min_child_samples': 10,
                'max_depth': 4,
                'learning_rate': 0.1,
                'colsample_bytree': 0.7,
                'class_weight': None,
                'random_state': 42,
                'verbose': -1  # To suppress output
            }
        }
    }

    # Dictionary to store results
    results = {}
    trained_models = {}

    print("=" * 60)
    print("MODEL TRAINING AND EVALUATION RESULTS")
    print("=" * 60)
    print(f"Training set size: {len(X_train)} samples")
    print(f"Test set size: {len(test_X)} samples")
    print("-" * 60)

    # Train and evaluate each model
    for model_name, config in model_configs.items():
        try:
            print(f"\nTraining {model_name}...")

            # Initialize model with hyperparameters
            model = config['model'](**config['params'])

            # Train the model
            model.fit(X_train, y_train)

            # Make predictions on test data
            y_pred = model.predict(test_X)

            # Calculate accuracy
            accuracy = accuracy_score(test_y, y_pred)

            # Store results
            results[model_name] = accuracy
            trained_models[model_name] = model

            print(f"✓ {model_name} - Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

        except Exception as e:
            print(f"✗ Error training {model_name}: {str(e)}")
            results[model_name] = 0.0

    # Create leaderboard
    leaderboard = pd.DataFrame(list(results.items()),
                              columns=['Model', 'Accuracy'])
    leaderboard['Accuracy_Percentage'] = leaderboard['Accuracy'] * 100
    leaderboard = leaderboard.sort_values('Accuracy', ascending=False).reset_index(drop=True)
    leaderboard['Rank'] = range(1, len(leaderboard) + 1)

    # Reorder columns for better display
    leaderboard = leaderboard[['Rank', 'Model', 'Accuracy', 'Accuracy_Percentage']]

    print("\n" + "=" * 60)
    print("FINAL LEADERBOARD - RANKED BY ACCURACY")
    print("=" * 60)

    # Print formatted leaderboard
    for idx, row in leaderboard.iterrows():
        rank_emoji = "1" if row['Rank'] == 1 else "2" if row['Rank'] == 2 else "3" if row['Rank'] == 3 else "."
        print(f"{rank_emoji} Rank {row['Rank']}: {row['Model']:<20} - {row['Accuracy']:.4f} ({row['Accuracy_Percentage']:.2f}%)")

    print("=" * 60)

    # Save top 3 models
    print("\nSAVING TOP 3 MODELS...")
    print("-" * 40)

    top_3_models = leaderboard.head(3)
    saved_models = {}

    for idx, row in top_3_models.iterrows():
        model_name = row['Model']
        rank = row['Rank']
        accuracy = row['Accuracy_Percentage']

        # Create well-formatted filename
        # Replace spaces with underscores and add rank and accuracy
        safe_model_name = model_name.replace(' ', '_').replace('.', '_')
        filename = f"rank_{rank}_{safe_model_name}_acc_{accuracy:.2f}%.pkl"

        # Get the trained model
        model_to_save = trained_models[model_name]

        try:
            # Save the model with pickle
            with open(filename, 'wb') as file:
                pickle.dump(model_to_save, file)

            saved_models[model_name] = filename
            rank_emoji = "1" if rank == 1 else "2" if rank == 2 else "."
            print(f"{rank_emoji} Rank {rank} - {model_name} saved to: {filename}")

        except Exception as e:
            print(f" Error saving {model_name}: {str(e)}")

    print("-" * 40)
    print(f"Successfully saved {len(saved_models)} models!")
    print("=" * 60)

    # Return leaderboard, trained models, and saved model filenames
    return leaderboard, trained_models, saved_models


In [ ]:
warnings.filterwarnings('ignore')

def train_and_evaluate_models_v1(X_train, y_train, test_X, test_y):
    """
    Version 1: Trains multiple models with specified hyperparameters and evaluates them on test data.

    Parameters:
    -----------
    X_train : array-like or DataFrame
        Training features
    y_train : array-like or Series
        Training target variable
    test_X : array-like or DataFrame
        Test features for evaluation
    test_y : array-like or Series
        Test target variable for evaluation

    Returns:
    --------
    tuple : (leaderboard, trained_models, saved_models)
        - leaderboard: pandas.DataFrame with model rankings and accuracies
        - trained_models: dict with all trained model objects
        - saved_models: dict with model names and their saved filenames
    """

    # Define model hyperparameters dictionary for Version 1
    model_configs = {
        'Random Forest': {
            'model': RandomForestClassifier,
            'params': {
                'n_estimators': 10,
                'min_samples_split': 80,
                'min_samples_leaf': 40,
                'min_impurity_decrease': 0.05,
                'max_samples': 0.6,
                'max_features': 0.3,
                'max_depth': 7,
                'ccp_alpha': 0.05,
                'random_state': 42,
                'n_jobs': -1
            }
        },
        'XGBoost': {
            'model': xgb.XGBClassifier,
            'params': {
                'subsample': 0.9,
                'scale_pos_weight': 1,
                'reg_lambda': 10,
                'reg_alpha': 0,
                'n_estimators': 200,
                'min_child_weight': 7,
                'max_depth': 8,
                'learning_rate': 0.15,
                'gamma': 0.5,
                'colsample_bytree': 0.6,
                'random_state': 42,
                'eval_metric': 'logloss'
            }
        },
        'Gradient Boosting': {
            'model': GradientBoostingClassifier,
            'params': {
                'subsample': 0.6,
                'n_estimators': 100,
                'min_samples_split': 80,
                'min_samples_leaf': 40,
                'min_impurity_decrease': 0.05,
                'max_features': 'log2',
                'max_depth': 2,
                'learning_rate': 0.01,
                'ccp_alpha': 0.0,
                'random_state': 42
            }
        },
        'CatBoost': {
            'model': cb.CatBoostClassifier,
            'params': {
                'subsample': 0.6,
                'min_data_in_leaf': 40,
                'learning_rate': 0.2,
                'l2_leaf_reg': 5,
                'iterations': 10,
                'depth': 4,
                'colsample_bylevel': 0.5,
                'class_weights': [1, 3],
                'border_count': 64,
                'bagging_temperature': 0,
                'random_state': 42,
                'verbose': False
            }
        }
    }

    # Dictionary to store results
    results = {}
    trained_models = {}

    print("=" * 60)
    print("MODEL TRAINING AND EVALUATION RESULTS - VERSION 1")
    print("=" * 60)
    print(f"Training set size: {len(X_train)} samples")
    print(f"Test set size: {len(test_X)} samples")
    print("-" * 60)

    # Train and evaluate each model
    for model_name, config in model_configs.items():
        try:
            print(f"\nTraining {model_name}...")

            # Initialize model with hyperparameters
            model = config['model'](**config['params'])

            # Train the model
            model.fit(X_train, y_train)

            # Make predictions on test data
            y_pred = model.predict(test_X)

            # Calculate accuracy
            accuracy = accuracy_score(test_y, y_pred)

            # Store results
            results[model_name] = accuracy
            trained_models[model_name] = model

            print(f"✓ {model_name} - Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

        except Exception as e:
            print(f"✗ Error training {model_name}: {str(e)}")
            results[model_name] = 0.0

    # Create leaderboard
    leaderboard = pd.DataFrame(list(results.items()),
                              columns=['Model', 'Accuracy'])
    leaderboard['Accuracy_Percentage'] = leaderboard['Accuracy'] * 100
    leaderboard = leaderboard.sort_values('Accuracy', ascending=False).reset_index(drop=True)
    leaderboard['Rank'] = range(1, len(leaderboard) + 1)

    # Reorder columns for better display
    leaderboard = leaderboard[['Rank', 'Model', 'Accuracy', 'Accuracy_Percentage']]

    print("\n" + "=" * 60)
    print("FINAL LEADERBOARD - RANKED BY ACCURACY (VERSION 1)")
    print("=" * 60)

    # Print formatted leaderboard
    for idx, row in leaderboard.iterrows():
        rank_emoji = "1" if row['Rank'] == 1 else "2" if row['Rank'] == 2 else "3" if row['Rank'] == 3 else "."
        print(f"{rank_emoji} Rank {row['Rank']}: {row['Model']:<20} - {row['Accuracy']:.4f} ({row['Accuracy_Percentage']:.2f}%)")

    print("=" * 60)

    # Save top 3 models
    print("\nSAVING TOP 3 MODELS...")
    print("-" * 40)

    top_3_models = leaderboard.head(3)
    saved_models = {}

    for idx, row in top_3_models.iterrows():
        model_name = row['Model']
        rank = row['Rank']
        accuracy = row['Accuracy_Percentage']

        # Create well-formatted filename
        safe_model_name = model_name.replace(' ', '_').replace('.', '_')
        filename = f"v1_rank_{rank}_{safe_model_name}_acc_{accuracy:.2f}%.pkl"

        # Get the trained model
        model_to_save = trained_models[model_name]

        try:
            # Save the model with pickle
            with open(filename, 'wb') as file:
                pickle.dump(model_to_save, file)

            saved_models[model_name] = filename
            rank_emoji = "1" if rank == 1 else "2" if rank == 2 else "."
            print(f"{rank_emoji} Rank {rank} - {model_name} saved to: {filename}")

        except Exception as e:
            print(f" Error saving {model_name}: {str(e)}")

    print("-" * 40)
    print(f" Successfully saved {len(saved_models)} models!")
    print("=" * 60)

    return leaderboard, trained_models, saved_models

In [ ]:
def train_and_evaluate_models_v2(X_train, y_train, test_X, test_y):
    """
    Version 2: Trains multiple models with specified hyperparameters and evaluates them on test data.

    Parameters:
    -----------
    X_train : array-like or DataFrame
        Training features
    y_train : array-like or Series
        Training target variable
    test_X : array-like or DataFrame
        Test features for evaluation
    test_y : array-like or Series
        Test target variable for evaluation

    Returns:
    --------
    tuple : (leaderboard, trained_models, saved_models)
        - leaderboard: pandas.DataFrame with model rankings and accuracies
        - trained_models: dict with all trained model objects
        - saved_models: dict with model names and their saved filenames
    """

    # Define model hyperparameters dictionary for Version 2
    model_configs = {
        'Random Forest': {
            'model': RandomForestClassifier,
            'params': {
                'n_estimators': 50,
                'min_samples_split': 40,
                'min_samples_leaf': 30,
                'min_impurity_decrease': 0.01,
                'max_samples': 0.8,
                'max_features': 'log2',
                'max_depth': 10,
                'ccp_alpha': 0.02,
                'random_state': 42,
                'n_jobs': -1
            }
        },
        'Logistic Regression': {
            'model': LogisticRegression,
            'params': {
                'solver': 'liblinear',
                'penalty': 'l2',
                'fit_intercept': False,
                'C': 1,
                'random_state': 42,
                'max_iter': 1000
            }
        },
        'Light GBM': {
            'model': lgb.LGBMClassifier,
            'params': {
                'subsample': 0.9,
                'reg_lambda': 0.1,
                'reg_alpha': 1,
                'num_leaves': 31,
                'n_estimators': 25,
                'min_split_gain': 1,
                'min_child_weight': 1,
                'min_child_samples': 10,
                'max_depth': 4,
                'learning_rate': 0.1,
                'colsample_bytree': 0.7,
                'class_weight': None,
                'random_state': 42,
                'verbose': -1
            }
        },
        'Gradient Boosting': {
            'model': GradientBoostingClassifier,
            'params': {
                'subsample': 0.8,
                'n_estimators': 10,
                'min_samples_split': 60,
                'min_samples_leaf': 20,
                'min_impurity_decrease': 0.05,
                'max_features': 0.3,
                'max_depth': 5,
                'learning_rate': 0.2,
                'ccp_alpha': 0.02,
                'random_state': 42
            }
        },
        'CatBoost': {
            'model': cb.CatBoostClassifier,
            'params': {
                'subsample': 0.6,
                'min_data_in_leaf': 40,
                'learning_rate': 0.05,
                'l2_leaf_reg': 3,
                'iterations': 10,
                'depth': 5,
                'colsample_bylevel': 0.5,
                'class_weights': [1, 1],
                'border_count': 128,
                'bagging_temperature': 0,
                'random_state': 42,
                'verbose': False
            }
        }
    }

    # Dictionary to store results
    results = {}
    trained_models = {}

    print("=" * 60)
    print("MODEL TRAINING AND EVALUATION RESULTS - VERSION 2")
    print("=" * 60)
    print(f"Training set size: {len(X_train)} samples")
    print(f"Test set size: {len(test_X)} samples")
    print("-" * 60)

    # Train and evaluate each model
    for model_name, config in model_configs.items():
        try:
            print(f"\nTraining {model_name}...")

            # Initialize model with hyperparameters
            model = config['model'](**config['params'])

            # Train the model
            model.fit(X_train, y_train)

            # Make predictions on test data
            y_pred = model.predict(test_X)

            # Calculate accuracy
            accuracy = accuracy_score(test_y, y_pred)

            # Store results
            results[model_name] = accuracy
            trained_models[model_name] = model

            print(f"✓ {model_name} - Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

        except Exception as e:
            print(f"✗ Error training {model_name}: {str(e)}")
            results[model_name] = 0.0

    # Create leaderboard
    leaderboard = pd.DataFrame(list(results.items()),
                              columns=['Model', 'Accuracy'])
    leaderboard['Accuracy_Percentage'] = leaderboard['Accuracy'] * 100
    leaderboard = leaderboard.sort_values('Accuracy', ascending=False).reset_index(drop=True)
    leaderboard['Rank'] = range(1, len(leaderboard) + 1)

    # Reorder columns for better display
    leaderboard = leaderboard[['Rank', 'Model', 'Accuracy', 'Accuracy_Percentage']]

    print("\n" + "=" * 60)
    print("FINAL LEADERBOARD - RANKED BY ACCURACY (VERSION 2)")
    print("=" * 60)

    # Print formatted leaderboard
    for idx, row in leaderboard.iterrows():
        rank_emoji = "1" if row['Rank'] == 1 else "2" if row['Rank'] == 2 else "3" if row['Rank'] == 3 else "."
        print(f"{rank_emoji} Rank {row['Rank']}: {row['Model']:<20} - {row['Accuracy']:.4f} ({row['Accuracy_Percentage']:.2f}%)")

    print("=" * 60)

    # Save top 3 models
    print("\nSAVING TOP 3 MODELS...")
    print("-" * 40)

    top_3_models = leaderboard.head(3)
    saved_models = {}

    for idx, row in top_3_models.iterrows():
        model_name = row['Model']
        rank = row['Rank']
        accuracy = row['Accuracy_Percentage']

        # Create well-formatted filename
        safe_model_name = model_name.replace(' ', '_').replace('.', '_')
        filename = f"v2_rank_{rank}_{safe_model_name}_acc_{accuracy:.2f}%.pkl"

        # Get the trained model
        model_to_save = trained_models[model_name]

        try:
            # Save the model with pickle
            with open(filename, 'wb') as file:
                pickle.dump(model_to_save, file)

            saved_models[model_name] = filename
            rank_emoji = "1" if rank == 1 else "2" if rank == 2 else "."
            print(f"{rank_emoji} Rank {rank} - {model_name} saved to: {filename}")

        except Exception as e:
            print(f"Error saving {model_name}: {str(e)}")

    print("-" * 40)
    print(f" Successfully saved {len(saved_models)} models!")
    print("=" * 60)

    return leaderboard, trained_models, saved_models

In [ ]:
leaderboard, models, saved_files = train_and_evaluate_models(X_train, y_train, X_out, y_out)
print("\nLeaderboard as DataFrame:")
print(leaderboard)
print("\nSaved model files:")
for model_name, filename in saved_files.items():
    print(f"- {model_name}: {filename}")

In [ ]:
#most imporant features
X_out_imp1 = df_out[feature_cols_imp1]
y_out_imp1 = df_out[target]

In [ ]:
print(X_out_imp1.head())

In [ ]:
for col in log_transform_cols_imp1:
  X_out_imp1[col] = np.log(X_out_imp1[col])

In [ ]:
print(X_out_imp1.head())

In [ ]:
# Identify numerical columns (excluding categorical and binned)
X_out_imp1[num_cols_imp1] = scaler_imp1.transform(X_out_imp1[num_cols_imp1])

In [ ]:
print(X_out_imp1.head())

In [ ]:
X_out_imp1 = pd.get_dummies(X_out_imp1, columns=cat_cols_imp1, drop_first=True)

# Ensure columns match in train and test
X_out_imp1 = X_out_imp1.reindex(columns=X_train_imp1.columns, fill_value=0)

In [ ]:
print(X_out_imp1.head())

In [ ]:
y_out_imp1 = le_imp1.transform(y_out_imp1)

print(le_imp1.classes_)  # To see which label is 0 and which is 1

y_out_imp1 = pd.Series(y_out_imp1)

In [ ]:
import catboost as cb

In [ ]:
leaderboard_imp1, models_imp1, saved_files_imp1 = train_and_evaluate_models_v1(X_train_imp1, y_train_imp1, X_out_imp1, y_out_imp1)
print("\nLeaderboard as DataFrame:")
print(leaderboard_imp1)
print("\nSaved model files:")
for model_name, filename in saved_files_imp1.items():
    print(f"- {model_name}: {filename}")

In [ ]:
# Exclude target and categorical columns from features
X_out_imp2 = df_clean[feature_cols_imp2]
y_out_imp2 = df_clean[target]

In [ ]:
print(X_out_imp1.head())

In [ ]:
for col in log_transform_cols_imp2:
  X_out_imp2[col] = np.log(X_out_imp2[col])

In [ ]:
print(X_out_imp1.head())

In [ ]:
# Identify numerical columns (excluding categorical and binned)
X_out_imp2[num_cols_imp2] = scaler_imp2.transform(X_out_imp2[num_cols_imp2])

In [ ]:
print(X_out_imp1.head())

In [ ]:
X_out_imp2 = pd.get_dummies(X_out_imp2, columns=cat_cols_imp2, drop_first=True)

# Ensure columns match in train and test
X_out_imp2 = X_out_imp2.reindex(columns=X_train_imp2.columns, fill_value=0)

In [ ]:
print(X_out_imp1.head())

In [ ]:
y_out_imp2 = le_imp2.transform(y_out_imp2)

print(le_imp2.classes_)  # To see which label is 0 and which is 1

y_out_imp2 = pd.Series(y_out_imp2)

In [ ]:
print(X_out_imp1.head())

In [ ]:
leaderboard_imp2, models_imp2, saved_files_imp2 = train_and_evaluate_models_v2(X_train_imp2, y_train_imp2, X_out_imp2, y_out_imp2)
print("\nLeaderboard as DataFrame:")
print(leaderboard_imp2)

print("\nSaved model files:")
for model_name, filename in saved_files_imp2.items():
    print(f"- {model_name}: {filename}")

In [ ]:
print(X_out.columns.tolist())

